# 3단계: 실제 빙하 데이터 기반 평가 및 Fine-tuning

**실행 순서**: `colab_train.ipynb` → `colab_iterative_train.ipynb` → **이 파일**

- NSIDC CDR G02202 해빙 농도 실데이터 활용
- NIC / Copernicus SAR 실제 빙산 위치 주입
- `colab_iterative_train.ipynb` 학습 결과 모델 로드 후 실환경 평가 + fine-tuning
- 결과를 Google Drive에 저장

In [ ]:
# ── CELL 1: GPU 확인 ──────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU: {result.stdout.strip()}')
else:
    print('GPU 없음 - CPU 모드로 진행')

import os
print(f'Python: {sys.version.split()[0]}')

In [ ]:
# ── CELL 2: 패키지 설치 ──────────────────────────────────────
import subprocess
pkgs = [
    'stable-baselines3==2.3.2',
    'gymnasium==0.29.1',
    'shimmy>=0.2.1',
    'torch',
    'numpy',
    'requests',
    'scipy',
    'matplotlib',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('패키지 설치 완료')

In [ ]:
# ── CELL 3: Google Drive 마운트 ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/arctic_rl'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/models', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/real_ice_data', exist_ok=True)
print(f'Drive 마운트 완료: {DRIVE_BASE}')

In [ ]:
# ── CELL 4: 모듈 생성 (config + land_mask) ──────────────────
import os
os.makedirs('/content/modules', exist_ok=True)

# config.py
with open('/content/modules/config.py', 'w') as f:
    f.write('''
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple

ROUTE_WAYPOINTS: Dict[str, List[Tuple[float, float]]] = {
    "NSR": [
        (35.10, 129.04), (41.78, 140.81), (45.65, 141.93),
        (52.00, 155.00), (63.00, 174.00), (65.77, 169.30),
        (71.00, 180.00), (73.50, 165.00), (76.00, 140.00),
        (77.60, 104.30), (76.00, 80.00),  (72.00, 55.00),
        (70.50, 30.00),  (62.00, 5.00),   (51.90, 4.50),
    ],
    "NWP": [
        (35.10, 129.04), (45.65, 141.93), (52.00, 155.00),
        (65.77, 169.30), (71.50, -156.00),(74.30, -118.00),
        (74.00, -95.00), (72.50, -80.00), (66.50, -61.00),
        (58.00, -45.00), (51.90, 4.50),
    ],
    "TSR": [
        (35.10, 129.04), (45.65, 141.93), (65.77, 169.30),
        (75.00, 180.00), (85.00, 160.00), (88.00, 0.00),
        (80.00, -10.00), (72.00, 0.00),   (51.90, 4.50),
    ],
}

MAX_SAFE_CONCENTRATION: Dict[str, float] = {
    "PC2": 0.95, "PC3": 0.9,  "PC4": 0.8, "PC5": 0.7,
    "PC6": 0.6,  "PC7": 0.5,  "IA Super": 0.7, "IA": 0.6,
    "IB": 0.5,   "IC": 0.4,   "None": 0.3,
}

ICE_CLASS_FACTORS: Dict[str, float] = {
    "PC2": 0.6, "PC3": 0.7, "PC4": 0.8, "PC5": 0.9,
    "PC6": 1.0, "PC7": 1.1, "IA Super": 0.8, "IA": 0.9,
    "IB": 1.0,  "IC": 1.1,  "None": 1.3,
}
''')

# rl_land_mask.py
with open('/content/modules/rl_land_mask.py', 'w') as f:
    f.write('''
from __future__ import annotations

_LAND_BOXES = [
    (59.0, 83.5, -73.0, -18.0),
    (63.0, 67.0, -25.0, -13.0),
    (74.0, 81.0, 10.0,  33.0),
    (70.0, 77.5, 51.0,  68.5),
    (78.0, 81.5, 91.0,  107.0),
    (79.5, 82.0, 44.0,  65.0),
    (73.0, 76.5, 136.0, 159.0),
    (70.5, 71.5, -179.0,-178.0),
    (70.5, 71.5, 178.5, 180.0),
    (72.0, 84.0, -90.0, -61.0),
    (68.0, 74.0, -115.0,-95.0),
    (65.0, 71.5, -165.0,-145.0),
    (58.0, 71.5, 5.0,   28.0),
    (73.0, 77.5, 92.0,  106.0),
    (69.5, 73.0, 67.0,  73.0),
    (64.5, 67.5, 172.0, 180.0),
    (51.5, 59.0, 160.5, 163.0),
    (43.5, 45.5, 141.5, 145.0),
    (46.0, 54.0, 141.8, 143.2),
]

_WAYPOINT_WHITELIST = [
    (45.65, 141.93, 0.5), (41.78, 140.81, 0.5),
    (74.30, -118.00, 1.5),(72.50, -80.00, 1.5),
    (66.50, -61.00, 1.5), (71.00, 180.00, 1.0),
    (71.00, -180.00, 1.0),(71.50, -156.00, 1.5),
    (65.77, 169.30, 0.5), (77.60, 104.30, 1.5),
    (72.00, 55.00, 1.5),  (76.00, 80.00, 1.5),
    (76.00, 140.00, 1.5), (73.50, 165.00, 1.5),
    (58.00, -45.00, 1.0), (62.00, 5.00, 1.5),
    (74.00, -95.00, 1.5),
]

class LandMask:
    def is_land(self, lat, lon):
        lon_n = ((lon + 180.0) % 360.0) - 180.0
        for wp_lat, wp_lon, radius in _WAYPOINT_WHITELIST:
            if abs(lat - wp_lat) <= radius and abs(lon_n - wp_lon) <= radius:
                return False
        for lat_min, lat_max, lon_min, lon_max in _LAND_BOXES:
            if lat_min <= lat <= lat_max and lon_min <= lon_n <= lon_max:
                return True
        return False
''')

print('config.py, rl_land_mask.py 생성 완료')

In [ ]:
# ── CELL 5: rl_ship_dynamics.py 생성 ────────────────────────
with open('/content/modules/rl_ship_dynamics.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math
from dataclasses import dataclass

KM_PER_DEG_LAT = 111.195

def km_per_deg_lon(lat: float) -> float:
    return KM_PER_DEG_LAT * math.cos(math.radians(lat))

def approx_dist_km(lat1, lon1, lat2, lon2):
    dlat = (lat2 - lat1) * KM_PER_DEG_LAT
    dlon = (lon2 - lon1) * km_per_deg_lon((lat1 + lat2) / 2)
    return math.sqrt(dlat**2 + dlon**2)

def bearing_deg(lat1, lon1, lat2, lon2):
    dlat = (lat2 - lat1) * KM_PER_DEG_LAT
    dlon = (lon2 - lon1) * km_per_deg_lon((lat1 + lat2) / 2)
    return math.degrees(math.atan2(dlon, dlat)) % 360

def normalize_angle(angle):
    while angle > 180:  angle -= 360
    while angle < -180: angle += 360
    return angle

@dataclass
class ShipParams:
    max_speed_knots: float = 14.0
    min_speed_knots: float = 3.0
    max_turn_rate_deg_per_min: float = 2.0
    length_m: float = 230.0
    beam_m: float = 32.0

    @classmethod
    def for_ship_type(cls, ship_type: str):
        cfg = {
            "bulk":      {"max_speed_knots": 13.0, "length_m": 220, "beam_m": 30},
            "tanker":    {"max_speed_knots": 12.0, "length_m": 250, "beam_m": 40},
            "container": {"max_speed_knots": 16.0, "length_m": 300, "beam_m": 40},
            "lng":       {"max_speed_knots": 14.0, "length_m": 280, "beam_m": 45},
        }
        return cls(**cfg.get(ship_type, {}))

@dataclass
class ShipState:
    lat: float
    lon: float
    heading_deg: float
    speed_knots: float

def step_ship(state: ShipState, heading_change_deg: float, speed_factor: float,
              params: ShipParams, dt_minutes: float = 10.0) -> ShipState:
    new_heading = (state.heading_deg + heading_change_deg) % 360
    max_turn = params.max_turn_rate_deg_per_min * dt_minutes
    actual_turn = max(-max_turn, min(max_turn, heading_change_deg))
    new_heading = (state.heading_deg + actual_turn) % 360
    target_speed = params.min_speed_knots + speed_factor * (params.max_speed_knots - params.min_speed_knots)
    new_speed = max(params.min_speed_knots, min(params.max_speed_knots, target_speed))
    dist_km = new_speed * 1.852 * dt_minutes / 60.0
    rad = math.radians(new_heading)
    new_lat = state.lat + (dist_km * math.cos(rad)) / KM_PER_DEG_LAT
    new_lon = state.lon + (dist_km * math.sin(rad)) / km_per_deg_lon(state.lat)
    return ShipState(lat=new_lat, lon=new_lon, heading_deg=new_heading, speed_knots=new_speed)
''')
print('rl_ship_dynamics.py 생성 완료')

In [ ]:
# ── CELL 6: rl_reward.py 생성 ────────────────────────────────
with open('/content/modules/rl_reward.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math
from dataclasses import dataclass
from .config import MAX_SAFE_CONCENTRATION, ICE_CLASS_FACTORS

@dataclass
class RewardWeights:
    collision: float = -50.0
    proximity: float = -0.5
    danger_zone: float = -1.0
    route_deviation: float = -0.5
    progress: float = 10.0
    smoothness: float = -0.01
    fuel: float = -0.005
    ice_concentration: float = -0.1
    episode_success: float = 200.0

@dataclass
class RewardContext:
    ship_lat: float
    ship_lon: float
    ship_speed_knots: float
    heading_change_deg: float
    speed_factor: float
    iceberg_distances_km: list
    iceberg_sizes_m: list
    cross_track_error_km: float
    along_track_progress: float
    max_allowed_deviation_km: float = 30.0
    ice_concentration: float = 0.0
    max_safe_concentration: float = 0.7
    visibility_km: float = 10.0
    wave_height_m: float = 1.0
    collision: bool = False
    episode_done_success: bool = False

def compute_dynamic_safety_radius(base_radius_km, speed_knots, visibility_km, ice_class_factor=1.0):
    speed_scale = max(0.5, speed_knots / 12.0)
    visibility_scale = 1.0 / max(visibility_km, 1.0)
    return base_radius_km * speed_scale * (1.0 + visibility_scale) * ice_class_factor

def compute_reward(ctx: RewardContext, weights: RewardWeights | None = None):
    if weights is None:
        weights = RewardWeights()
    components = {}
    components["collision"] = weights.collision if ctx.collision else 0.0
    prox_total = 0.0
    danger_total = 0.0
    for dist_km, size_m in zip(ctx.iceberg_distances_km, ctx.iceberg_sizes_m):
        base_r = max(0.5, size_m / 1000.0)
        safety_r = compute_dynamic_safety_radius(base_r, ctx.ship_speed_knots, ctx.visibility_km)
        if dist_km < safety_r * 3:
            prox_total += weights.proximity * math.exp(-dist_km / max(safety_r, 0.1))
        if dist_km < safety_r:
            danger_total += weights.danger_zone * (1.0 - dist_km / max(safety_r, 0.1))
    components["proximity"] = prox_total
    components["danger_zone"] = danger_total
    dev_ratio = min(1.0, abs(ctx.cross_track_error_km) / max(ctx.max_allowed_deviation_km, 1.0))
    components["route_deviation"] = weights.route_deviation * dev_ratio
    components["progress"] = weights.progress * max(0.0, ctx.along_track_progress)
    components["smoothness"] = weights.smoothness * (abs(ctx.heading_change_deg) / 30.0)
    components["fuel"] = weights.fuel * ctx.speed_factor
    excess = max(0.0, ctx.ice_concentration - ctx.max_safe_concentration)
    components["ice_concentration"] = weights.ice_concentration * excess * 10.0
    components["episode_success"] = weights.episode_success if ctx.episode_done_success else 0.0
    total = sum(components.values())
    return total, components
''')
print('rl_reward.py 생성 완료')

In [ ]:
# ── CELL 7: rl_environment.py 생성 ───────────────────────────
with open('/content/modules/rl_environment.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math, random
from typing import Any
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from .rl_ship_dynamics import (ShipState, ShipParams, step_ship,
    approx_dist_km, bearing_deg, normalize_angle, KM_PER_DEG_LAT, km_per_deg_lon)
from .rl_reward import RewardContext, RewardWeights, compute_reward, compute_dynamic_safety_radius
from .rl_land_mask import LandMask
from .config import ROUTE_WAYPOINTS, MAX_SAFE_CONCENTRATION

class Iceberg:
    __slots__ = ("lat", "lon", "length_m", "width_m")
    def __init__(self, lat, lon, length_m=5000.0, width_m=3000.0):
        self.lat = lat; self.lon = lon
        self.length_m = length_m; self.width_m = width_m

def _random_icebergs_along_segment(lat1, lon1, lat2, lon2, count, spread_km=30.0):
    bergs = []
    for _ in range(count):
        t = random.random()
        c_lat = lat1 + t*(lat2-lat1)
        c_lon = lon1 + t*(lon2-lon1)
        off_lat = random.gauss(0, spread_km/KM_PER_DEG_LAT/3)
        off_lon = random.gauss(0, spread_km/max(1, km_per_deg_lon(c_lat))/3)
        st = random.choices(["small","medium","large","tabular"], weights=[.4,.3,.2,.1])[0]
        sz = {"small":(random.uniform(25,80),random.uniform(15,50)),
              "medium":(random.uniform(80,200),random.uniform(50,120)),
              "large":(random.uniform(200,500),random.uniform(100,300)),
              "tabular":(random.uniform(500,2000),random.uniform(300,1000))}
        lm, wm = sz[st]
        bergs.append(Iceberg(c_lat+off_lat, c_lon+off_lon, lm, wm))
    return bergs

def _cross_track_error(ship_lat, ship_lon, wp1, wp2):
    lat1,lon1=wp1; lat2,lon2=wp2
    dAB_n=(lat2-lat1)*KM_PER_DEG_LAT
    dAB_e=(lon2-lon1)*km_per_deg_lon((lat1+lat2)/2)
    dAP_n=(ship_lat-lat1)*KM_PER_DEG_LAT
    dAP_e=(ship_lon-lon1)*km_per_deg_lon((lat1+ship_lat)/2)
    AB_len=math.sqrt(dAB_n**2+dAB_e**2)
    if AB_len<1e-6: return approx_dist_km(ship_lat,ship_lon,lat1,lon1)
    return (dAP_e*dAB_n-dAP_n*dAB_e)/AB_len

def _along_track_fraction(ship_lat, ship_lon, wp1, wp2):
    lat1,lon1=wp1; lat2,lon2=wp2
    dAB_n=(lat2-lat1)*KM_PER_DEG_LAT
    dAB_e=(lon2-lon1)*km_per_deg_lon((lat1+lat2)/2)
    dAP_n=(ship_lat-lat1)*KM_PER_DEG_LAT
    dAP_e=(ship_lon-lon1)*km_per_deg_lon((lat1+ship_lat)/2)
    AB_len2=dAB_n**2+dAB_e**2
    if AB_len2<1e-12: return 0.0
    return max(0.0,min(1.0,(dAP_n*dAB_n+dAP_e*dAB_e)/AB_len2))

class IcebergAvoidanceEnv(gym.Env):
    metadata={"render_modes":[]}
    OBS_DIM=22
    N_ICEBERG_OBS=3

    def __init__(self, route="NSR", ice_class="PC7", ship_type="bulk",
                 difficulty="medium", reward_weights=None,
                 real_icebergs=None, real_ice_concentration=None):
        super().__init__()
        self.route=route; self.ice_class=ice_class; self.ship_type=ship_type
        self.difficulty=difficulty
        self.weights=reward_weights if reward_weights else RewardWeights()
        self.waypoints=ROUTE_WAYPOINTS[route]
        self.ship_params=ShipParams.for_ship_type(ship_type)
        self.land_mask=LandMask()
        self.max_safe_conc=MAX_SAFE_CONCENTRATION.get(ice_class,0.5)
        # 실데이터 주입
        self.real_icebergs=real_icebergs  # list of Iceberg or None
        self.real_ice_concentration=real_ice_concentration  # float or None
        self.action_space=spaces.Box(np.array([-1.0,-1.0]),np.array([1.0,1.0]),dtype=np.float32)
        self.observation_space=spaces.Box(-np.inf,np.inf,shape=(self.OBS_DIM,),dtype=np.float32)
        self._difficulty_cfg={
            "easy":  {"n_range":(0,0),  "spread":20},
            "medium":{"n_range":(3,8),  "spread":30},
            "hard":  {"n_range":(8,20), "spread":25},
        }
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        seg_idx=random.randint(0,len(self.waypoints)-2)
        self._wp1=self.waypoints[seg_idx]
        self._wp2=self.waypoints[seg_idx+1]
        self._dest=self.waypoints[-1]
        init_lat=self._wp1[0]; init_lon=self._wp1[1]
        init_hdg=bearing_deg(init_lat,init_lon,self._wp2[0],self._wp2[1])
        self._ship=ShipState(init_lat,init_lon,init_hdg,10.0)
        # 빙산: 실데이터 우선, 없으면 랜덤 생성
        if self.real_icebergs is not None:
            route_lats=[wp[0] for wp in self.waypoints]
            route_lons=[wp[1] for wp in self.waypoints]
            lat_min,lat_max=min(route_lats)-5,max(route_lats)+5
            lon_min,lon_max=min(route_lons)-10,max(route_lons)+10
            self._icebergs=[b for b in self.real_icebergs
                            if lat_min<=b.lat<=lat_max and lon_min<=b.lon<=lon_max]
            if not self._icebergs:
                self._icebergs=random.sample(self.real_icebergs, min(5,len(self.real_icebergs)))
        else:
            cfg=self._difficulty_cfg[self.difficulty]
            n=random.randint(*cfg["n_range"])
            self._icebergs=_random_icebergs_along_segment(
                *self._wp1,*self._wp2,n,cfg["spread"])
        # 해빙 농도: 실데이터 우선
        if self.real_ice_concentration is not None:
            self._ice_conc=float(self.real_ice_concentration)
        else:
            self._ice_conc=random.uniform(0.0,0.9)
        self._step=0
        self._max_steps=500
        self._prev_along=0.0
        self._total_reward=0.0
        self._collision_count=0
        return self._obs(), {}

    def _obs(self):
        s=self._ship
        wp2=self._wp2; dest=self._dest
        dist_wp2=approx_dist_km(s.lat,s.lon,*wp2)
        bear_wp2=bearing_deg(s.lat,s.lon,*wp2)
        dist_dest=approx_dist_km(s.lat,s.lon,*dest)
        bear_dest=bearing_deg(s.lat,s.lon,*dest)
        cte=_cross_track_error(s.lat,s.lon,self._wp1,wp2)
        along=_along_track_fraction(s.lat,s.lon,self._wp1,wp2)
        dists=sorted([
            (approx_dist_km(s.lat,s.lon,b.lat,b.lon),b)
            for b in self._icebergs
        ],key=lambda x:x[0])[:self.N_ICEBERG_OBS]
        berg_obs=[]
        for i in range(self.N_ICEBERG_OBS):
            if i<len(dists):
                d,b=dists[i]
                berg_obs+=[d/100.0,
                            math.log1p(b.length_m)/10.0,
                            bearing_deg(s.lat,s.lon,b.lat,b.lon)/180.0-1.0]
            else:
                berg_obs+=[1.0,0.0,0.0]
        obs=np.array([
            s.lat/90.0, s.lon/180.0,
            s.heading_deg/180.0-1.0,
            s.speed_knots/14.0,
            dist_wp2/1000.0, bear_wp2/180.0-1.0,
            dist_dest/5000.0, bear_dest/180.0-1.0,
            cte/50.0, along,
            self._ice_conc,
            len(self._icebergs)/20.0,
        ]+berg_obs, dtype=np.float32)
        return obs[:self.OBS_DIM]

    def step(self, action):
        hdg_chg=float(action[0])*30.0
        spd_fac=float((action[1]+1.0)/2.0)
        self._ship=step_ship(self._ship,hdg_chg,spd_fac,self._ship_params_obj if hasattr(self,"_ship_params_obj") else self.ship_params)
        self._step+=1
        s=self._ship
        # 충돌 판정
        dists_km=[approx_dist_km(s.lat,s.lon,b.lat,b.lon) for b in self._icebergs]
        sizes=[max(b.length_m,b.width_m) for b in self._icebergs]
        collision=any(d<(sz/1000.0*0.5) for d,sz in zip(dists_km,sizes))
        land_hit=self.land_mask.is_land(s.lat,s.lon)
        along=_along_track_fraction(s.lat,s.lon,self._wp1,self._wp2)
        cte=_cross_track_error(s.lat,s.lon,self._wp1,self._wp2)
        done_success=(along>=0.98 or approx_dist_km(s.lat,s.lon,*self._dest)<5.0)
        ctx=RewardContext(
            ship_lat=s.lat, ship_lon=s.lon,
            ship_speed_knots=s.speed_knots,
            heading_change_deg=hdg_chg, speed_factor=spd_fac,
            iceberg_distances_km=dists_km, iceberg_sizes_m=sizes,
            cross_track_error_km=cte,
            along_track_progress=along-self._prev_along,
            ice_concentration=self._ice_conc,
            max_safe_concentration=self.max_safe_conc,
            collision=collision or land_hit,
            episode_done_success=done_success,
        )
        reward,_=compute_reward(ctx,self.weights)
        self._prev_along=along
        terminated=collision or land_hit or done_success
        truncated=(self._step>=self._max_steps)
        info={"collision":collision or land_hit,"success":done_success,
              "along":along,"cte":cte,"step":self._step}
        return self._obs(),reward,terminated,truncated,info
''')
print('rl_environment.py 생성 완료')

In [ ]:
# ── CELL 8: __init__.py 생성 및 모듈 임포트 확인 ────────────
with open('/content/modules/__init__.py', 'w') as f:
    f.write('from .config import ROUTE_WAYPOINTS, MAX_SAFE_CONCENTRATION, ICE_CLASS_FACTORS\n')

import sys
sys.path.insert(0, '/content')

from modules.rl_environment import IcebergAvoidanceEnv, Iceberg
from modules.rl_reward import RewardWeights
from modules.config import ROUTE_WAYPOINTS
print('모듈 임포트 OK')

# 환경 검증
env = IcebergAvoidanceEnv(route='NSR', ice_class='PC7', ship_type='bulk')
obs, _ = env.reset()
print(f'obs shape: {obs.shape}, action space: {env.action_space}')

In [ ]:
# ── CELL 9: 실제 빙하 데이터 로드 (인라인 embed) ────────────
# 모든 실데이터가 이 셀에 포함되어 있습니다. 별도 파일 업로드 불필요.
import json, base64, gzip, os, datetime
from modules.rl_environment import Iceberg

DRIVE_BASE = '/content/drive/MyDrive/arctic_rl'
DATA_DIR = f'{DRIVE_BASE}/real_ice_data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── 인라인 데이터 (gzip+base64) ──────────────────────────────
_BERG_B64 = """H4sIABs88GkC/81bXWsbRxR9D+Q/CD/Ho7lf89E3S6aJKIRA8EMpJSi2cA3+CLbcUkL+e0eOW0ues1QbpPX4xXjXVs65c+6ZM3c3X1+/Go0O7m7ub08XBz+NDk4+jt7Plxc31/PL0ex0MZourpeL29F4NPn1ZPz+6OPR6M+L+ejtxfLd/edy9f3H2fF0NJt9OHjz8EFn8+XDx7DncOj1kOLjjc+L2/NPpzf318tyO8Wni3fl599WP41GX79/KzcuzlYfchTD9PufP1y8vLkuVw8lOJG1q/PVJx5adrZ2cXF9vvzj01W5wzkIP9356+Ls8QaVO/p0Y/n3lwfkl/Pb88X6vzq/W366//IvMa9j8uMVvYPvv/LtTQf4RDV2C+6R+zr2oC5mCN4oWUDgNYj3Nfjl/PN9IbAT+ALgs4uphh8dJVz78qUvUnutwZNXl3KFPopLoSf68sX7RG+g9NFxDT4kJwTBF+GwR+DN1hW1a+wTnycVeFJxTDX44HyH7L33uPKbrHYv+4L/bYVfyZmi2nNTqp8wHwHZx+QIyN67gA3TJPr0Ip5TCPxciyfmTXf8z3Ssw3Qsc3qZ8r8D6JPzXFefnWLTIU0UepjO1eLs4v5qB/CnZLBxPWhc67TMztrjxt1V7aeUatvR6FSR6+TUzzKjrjfEHrDXKUfVBYI7beiHvTTzHjU/Zarrnos8FIWc3DclbPbC7sHXKSEH1K4F+wajNexeIg+/zU7FI+icYLbskHsIIW0PfXdGI4Swa0bYQ2vYAzDJAj7CDQpHMy0uCS3+2VFl9xvsVHIFP5rzQDa502m6Nijskjsr/TFZnW4SuczI4oNA8FnVYLuqj6r7LH2BP0Hw1+X6pByP4a/MHLr8MwvdC/x6k0recYQbbGgqmR1zfZaNxRcJyV5x6Z9BHC6XHQvXob5EmxBQ5SX088v9o5cjeJhFcwRzjI8koRQ/vch5sOCfAPwlFqO23eiGtbZNKfPLtK0ImKBZOREaKn/E2ucYk3+JvpU6oCV23noE486AtudweSxgiKMJmo45Sf3Gfx1tu0vhxAr+avBqyDIZ55xVFusjmx0OL6OiSUgqZ6p6vypmlELsnEPBvvX+Bw6zMmYbbzH9A36fPYQekks0BHQd+/T/0CclogG3FH6G8rHu5JINU/Y43mZ68xZIxrLzoSPcN1R45hma/MWSxKBqJElD4DPATkXxSdC0Plgz0KcAejG3cqAS5PI6iGR4K5uZSp2JyZNjJJgSzBpS+1TQpJJdNpSIW0J+LB74o5oDWd47G6RJiz3mbZDXg48USooXWPN2WvTkFwFjPtKMEnx45pf7qzmF7aBHIBd1iv2FZSBXj9uBT7DuGuDMQJqJArPZh8MSMumQvFeQ4gM50er4unpAq9xFooNDTeHuan552U0hjymuEjD14GDgSX6xl5hqDqHsrtYihwhCcclmtZQKBTNpkYLHUlIgJPVDMMi9GQTcDAabwWKLHMjDZsj1BN9WMwVukQN7rCQCFEysSQqMpRQMSYmblBLjdqhftigUPDXZ0dKhJB+hKzUpJTAkfJBShkpqkYFGyIBJwO6WY5OrYIQ5qMBuaJFCFNzPyJI2w1M7FBTvbjDq5dikrUaDHEJGWS/lJrshho7IjShQmxQ6TClAKYUmpeTRO8zmDEhJnPAAWzT73tsblhKjZWBtkkLI8PAmwFnNBWmSQ0zwMbaXWkq6+QrZvihQbyVRx/4GY4bEJjkw7AaKhtpBmlwH9LyjrIMnFJWEhlgH7t0OuSN0V+2wWp3cJAfqWAc8zohDaEl694PB1ztiRtYaqU0OCa6DCQrekqxFDsnDdQgJ+VKg1CIHQ+sgrp4qJZcHyUq9GQTFDMAmnR1RkxQCpOADAwqem2wGYmxKKLJSm55EAil4ZKucmmxn9TCxKjj8WAmBTS5DSHAZNl7pe+IwxCH0B2wVzTMYvPhdkpIfQkm9n1wpQQYBHBxS1zuku6WQei8C3hm8GdoZrEkOGmBDg0cmq/+uZE1SwMsg4OiTh3n61p8DGeQQFXHQARhsjsVW335//erbPyWCeNjZQQAA"""

_COP_B64 = """H4sIABs88GkC/82dT89nR06F90h8h1ZWINGX8p8qV7FDiDVoWLBALMJMhEZCCUpmVgg+O1W/zmTy5j4XmgX9nlUmb5Lu0x5fH9tlH//Hn/7Jhw9f/fbX3/zLN9//6w9f/dWHfzo/+PDhPz795fyz3+yffvU3f/f3H//hr3/1sbVmX/3FT//w377+3f6nZZeV5c9+/N23+8deVzb7409/+O733//6m9ev9t2/f/P9t7/99e9/+LB/0Q9/9quvf/P19z98/bs//9kvvf+N3373+r29+fjY7KPPD3/54dPf+P77rz79q//5F/8rYifEbbndEI8rRu8CkAMh95gIWcHIyYgRbwsBwP0OuF1r2kDIawhAHgi5Flo5BQAXAZ7D77FiXqOVgo0nQR6tO0CO+vmP3w3yIsiZt/AWtp2l3h+wtRvgsS6vm1d87HmZT4GAbMaQ7QHzUMDshNliTMKcLQUihgVhbt3ukMdluZYA5ETIq6GZl5UA+VlHzJXgzv2ysRTcebCdY+EnaAqpp90JcMxrBflGXWtNAQa0iZhnH/QN5moCFGiLMWeQP69QCM/eEHNGJzvPruDPboi5d/oEW4WAO7sT5O3NRWYeYQqYAzFXYHjOcgXXSLazGWGOXVEJYO6IeWdOhNlrKPjGIMwVgeF5TIXC1ZEGx6qJmFcIFCmONDjKgjBXLwEadKTBHR4wRWqpUL0G0mC1Kq5SukD6HEiDY/bAlN8VGhuBPDh6wxSpWhOoBgN5cOwUlb7B1U0gbkSynedE7u4KkJEGR3bnalAg24jBVnanL3BaKEQ6ZMGeragYzPFzdnw3zMiCPQqzjRUKkJEE+yAS7Kd+FcCcSIK9r8HJs4KdE0mwe8eEf7zpR78bZiTBLEM7N1NI+DMYcxiGuqaQPCeSYPaOSV1MCTsjC2Z0bPHPqdATTaTBDRn9eZtZ4RtEGoxcnTAr5EeJJBgDuzG7QBHInBNJ0NfCh5Qh0eDvSII+uVMXofBe1ZEEo3GuYVPAm7uzmR1do5lCf78jB/rAXkz2pmBmpECbC51ZgQA7EqBHw65GzS7Qwe1IgLYWvkhUdAU7IwFaFdeubzoH74YZKdAGv/x0VzAzUGBdK+lNMK65mgDm0QhzTay2rSk8Fw9DyGNhz3mUCXjzcMY8sTwZptCJGUGYox7s/KYT9m6YgQPHtZbz0FQJVCejE+Rea1LUGKkQnMdAzD0Rc7kE5iLM7tRz3phXV/CNSZhbLJzmqTSBAmUAC/ZrYuocl6XCsGU1wpyLJmM25qXQ3C8jzNFoQNQ3pSjQYDlh9qTntY25FGaHKwizJUMeCu2jSoLcit5Q/IppAi/c1REzNnB95/uhEDaABvNacyLmLIVJgirGTEOtB/NUCHUTMb/pYfwRs0s092sR5jnpEWVjHingz7Mh5mLf8KFQDU5DzPjG/bKzwDc4HTHHZMgh0PaawZD7g5kVBkRnImbv+RA2FFyjE+Zdc7Od37wVvhtmpMGaK9g3FGhwIg2WhxNmW0sBM9Jg2RqMeQhQ91yMuRZjVqhSFtJgH8X+HAKhbiELZhgz93IBd17IgmmF4TkUpkxWPEBmQhEAjBQYfXAW2hVSjQUUGNckV7arusIL9xoEuSYtvtr+o6SCaxRiXpQd2dUlJi3XJMy9RRDmt9Nf74Z5MWaaANyYBVzDWiPIgW/cdnkT6B8ZqMQczJPmszfmEqhdDXRiXpiJTzbmqWDmQMjlGOncBPblDZRiDubg6Ox9pADm/oB5MGaJTxBZMNpkO5cp2LkeMCML+hDY47aGLOgrjX3DSwAzsqDjjv/GHArh2Rpjfoh1sVwAM9KgVz5gFqiqzJAGt/mRBs1MADLS4Nn9v0NuuxAoBcxAg37NRS8S7RrRFdy5I+ZGL2ztUhjDNVKNeUHuBDlnV4h0RZjHpFbMxjxMwc6TMPecQZhDoU9gpBqzMUdHzNkUQh2pxhzME0NdDIVQR6oxZ2hgGvvGEPANko3xK8zQN3ZRI1BZkWxMu87q/w1z9u0bIVClgGxMX9dahPkMRihUsCQbs32g93tWl36tEpipM5CNOXY2WGLbvtGWQLvcSDbmfGtk553uWVfAPMnOO63gb7A1Bd9YiHn4om/w7ev3e2EG2ZiXnR3jxptXrHeDbAg5QXTxQJ5dwczO4fku7HxCXU+BMW0D1ZhjZ0NvtiEgRGaRDBkEsl7eLKBZYaAa88LM3lwKLdEYHOgIc1ypEDMKETcY2NiIt5EVYgZSYHVQqNuYl8DosAUyYBuJ2VGYQg8XNGP63E4LfbocVymIvRloxlQc3du7O28vnwJ3AgwkY2on9T3vDDivGq4AORAylFRHHFBgPNtAL+Zo3pKWkI15vRHofDfMnTAXpBk26jizAJ2AXsyxc/rdmXeScYWnQqArAt0zFzuHwLCzgWRMP+uNkGnY2W3rqeDRi0DngB6unQaeCShAGKjGHNmSDs+C9pqRF/gMQTXmyMMMco44Ez0KdnbC7GnkHHaEnwWYEHRjXgqWZWjp3paCdwAZ5s440KPtWk1g/9VAPOaIYVWypTOngk8DHdbl1N04Ac9NgcNBPuaAnr4YdFNo14F+zAE9FlOLLVewNBxW2hneuEtT2yUg8WygHlO/EGX9Ee4ux0NhfhHEY2qTSmv3I2x5hcI1QQPtmDpphd1iho3LSkBrw0A6ZkMuuPZjRyRSwsqJkPMurHesbArRApRjKnYNex8f3p+fd4VOHQjHHMhvxtB+gpxDAnJRb6PfpUF8l69TwS0mAvYgwKnQPgLJmI047zO4G7FHChAfCMa8EDcnyApUDXIxdXbU7vdd/dxLFfjwCvugUY3upa4hcA7FCvugbdzP+/hOOrqEX+AZ3e3I948vjnyCAFWDVEydTUAnK0+FSXIQitmIx6J70E1Ba99AJuYg7vdm84GscCPHQCXmQLa7XuFxiwoFT14I2Qc5hkl8fKARsyGHNyISV5Amt4kH5MPuqf0LsgBgvB/vCdmFnfc0AcR4Ph76Fr6uMoVRv4m8Z+teovpLakrh0+sMuRPvmcTa18QTumH3ta+IXaIoFCMTmW8OmHJ4zTovhc8Pqa/ePC78YXZ8Xc0VcqKJ3NcHqWXX5muFWdvVOCsiBZ5zpVYBMpJfOUFeO/VQWEdayH/daH1jXukKz1GLGRAlkddxGYHovJADs1PYmNeOdAqYkQQHlNiv8FwKxL2w/EuHtYLjz6GwQrWQBgMPnM/LwgVeVxfSYIzEuGGmsEK1FmN2/Aa9BN5KvHENuGBCcWMeAm8l3pAGswV+glUCb5XenF0jijBPgbNJ3pAF880o4h9ZcNVSMHM+mBmZ2xSSZ2/Igm4z2TUkogayoNPS17azv+mOvhtmZEF3OgywC6spkCF5QxY0VETeoa4LzFR6QxbEBddTWDWBDpIbsmBDSZt5dYWHCDekwbY4Q9qfoEB8NqTBhhogJz4LhA2Qh6l2rQw2s4I2hYM8zMFcAxP+pbDd44a3k8asorstPQTGmt3wdtKwwjtEo1kXwIz6MBYQ6nKdiXeFsIH6MG0GYq6pEDZAHsb2J0jbouvqTWBbzUkeZmMeIMl6ME8BfVMneZiNmU5UHdcwgRa/kzzMwQz9xYNZoB3jpA5zIDuaeUyBlqiDOswLMzD3y8xd4RPshHkOusneLp+lYOdBmLM3UNM4Cf9S8OdizOQZbSkQCmjDHMSkfHswuwJvO5JgekMStKbgzYEkmKRgeTALyLp5IAdmq2DIQwEzcmAsdmdzhQ5SIAlGkdTR+QRDwZ2RBCOTg7OVQNiIzpjbZMwC79weSIINtp835BwKiFEgbWfOnSB7F5imc1CH2Zhnh/TomDkUerggD3Mwx0RvzhS40uggD/PCPNidrRQwG2MOxuwKLAj6MAezgzrFxrxdRsHOKBRac7E/T4WyCiRiXpirc3gW2ObwRKHQisbxWYIFQSLmYO4N/TmrBDpIiTw4WkLCX1cvAe0BT+TBPsnOB7OAgqUn8mCnS9YvzALuDPIwc7vzos3AUQIH1xzEYTbiWfeK+6zxN4EdVwdtmFHnserepoud0glc5nBQhnlBhkHcvFximgd0YY6wGLyunZuCCu/bIApzPr68q35YXN0USiqQhNmQx31rbSPe8UIBcSHifm/DbMg1FBqhfT4IFA4QpIguULV23Ikv91uEszrj5AJGHrgUD9fjt41HKaT4A5fiExbttpHbEFhm9IFb8Wb3jGiT9WhdATJuxb89wfiTY6wmoELuI3mh6t4BtXEe4hWsjCOhWRAwxjVSYXZg8ERoA8jzMhe4veekBrNz5XVvM0ffjjEUrDwJcqv7QaoN2UzBlRchNrt35SJ2wBC4aOEkCHMgA+BhoQDY0CuyL4DclsIrKwnCtMvhQlKcT1IhXBQOgjZYf97OMhTehUEPZhxl9LuoWNi1TGGiEvRgXpCdELeuYGRci5+QEb0gC2jEeiHxtfszdrz8W6BMBT2YY2S4ChEvLR4FIy+CvL+yAToaSyIhAj2YDblWoxDXpoKVJ4phz7w3Az7a646IwjDlRDXsZTAWZWflKwQu9vhENeyetVBYus9UcI9k0B0xjzc3f98NM4phZ8tk0KGQM4M4zAYdyx5Am8BevIM8zAE9R38CrWDpyaAjHkALXOFwEIh5gbYnSwus8zsoxJzgsXpHbjEFpU0HjZij/18z8aKFuUKYBpGYA3qtifr/EQpPraAS8wKNPr1Bu0IzdCXftHg4DxE2BFKP9XAeojh69DYEaByEYo57zM75UnWF2S5QivkE+gGzKWCejPne5fgE+k3G+m6gkRDPXTAEnSGwtxagFXMsTXpNL0sPAeWVaEiIcxVfPLEUSD2i+UPEg7L2DFEpZKbRmBBpDeVYuoXA9F80vJc0BzUQjnu4gCZBgGTMAT0bs/hcU8E9kBCpUfrjZaqmED2K3WPEE2iFOI2MOOHOxSf3EKgQA1Rj+uvgHrP4CoEt0jAkxAlvQp+CRxc46BOGhFjliaCjBPq8YVgh1noAbVNAaiMsGLQ93C1TmKMKY0L0p0uNClHakA9HSy7F++wKLj0YdJsPoAWuvIYVBw97+A6XCXCLMR82rsSnCUw/hy029OSypQ0ToBZHPiwuEM91SROwtDMfjgfIAqs04cyGfT7Y2SWcIx5CB99LbUshlfZ88OiHDC8F9ufDkQ5rPlD4ErhFE45sWBlP36FCSevMhr0/9JYUbiuFT3bpZVxp2VIIHkyH8XQCeAgIUEU80GF/6HiYgnsE02Ha5DBdAhcbIpgQ2zSOHiGwYxOBhDjWQ4oXCtuaEUyIb+6Ev/kQu4J7cH04k91jp4QC5BJcH66HfMm6gBxjBDOij4daqyuk0zHZPWo++LQrxOnFoMcTIb4/5EQ+fLs++PMonQJfYSIdDh4h9DOmJ/AVpjNofwKtYGhmw9EfKLwJ3MSLzAdDM4V7KbxaJLNhhD94h0KMzvEA+vE7FOgeZP2fgoebwCHbSGZDf2h5hMK5qEhmQ5vzAbSAWlJ05kOLh1xaQfM+OhOi5UNjugSWF6IjIQY7x4qhABn50CcmS3FlKtThoDJTcXXK8HqdgXoFQ3fEHHRjZ2MOhVErEJo5mB0R965g5WLEtgizRtCYCPnNXNIbZxaAvAhyDkfP6FOhtUtSMxtzp5ttL88QKL5BbGYcsSHY0Yo4d4wEYgaozYzBnTtf11Q4hxAgNzPODbz7CYeoKwX2b2Kw0qh3VDm0pfAyO1hptJGiax1Na4F6cKDS6IgOgnZjVy9NATMrjSa0oTdmhZCBMqNYCub4xXPLu2FGmdFBb7Ibs0+FtLkaOwbMzbwwKzRzyz5fNnecUR+BD7BQbjsKdkGyX13hll8Uym37gsbXxpxLoXYtJEGn/YRj56YwNFNIgj5Inrjvb1NggT0KSdA7XeXqZ0tIwTeKMceDbyiYGWnQE66WHsghoOUShTQYk+6+vETB3h/yRN21mPcXoNMIKwFV5QD5mXG0+EG7+hwAHaFgZifMZm0Q5nKFpG5iHdgWXEw8mHMoYEbF7eXw+HMwSyR1oD1zMDe6RR/nFoVAUgfSM596BHcZz3n0oBXMDO+C8+pl2CTIUmjTke5MHWkA5zNGpWDnRZgn3z7zJiDIHiQ6U9dYkYS5K1zXCdCcOQtLAWWV709QInUGyZnXEyscRDiSgk3hQQIUZ16SgpCHWjvaLQq+gaXgsAGYaztHU6DBxbUgLZtW34FD4iPkWhBefj5hlnBo4MHt0DggX7VLAYE7c0F6M3PnyfCSaechywUOJ8ZCJuyBOgbzjEC8v6WzIRc2GoB+ac0LvLIlyc0MEjB+Ye5vYsq7YXbS7h+wPRFnEExgsi5Ba6Zel6XvGsZn78oUvBnvULQFdcoRqhWYI07QmamxP8BAlfYp8P6aoDKz6S7inm9sK+ccClYugmxZCQXsFEjqEgRmTsDwe8KfcS0XGBBN0Jc5kK1PgLyNLGBlUJfZkG3cN/GyXasJ6JYlaMscyH7vh27I2QW0tNKQ/BpcxjuQS2AHLw3Jr42BkBXU8NOY/O7VyUYcHgqu3PE6EAxr5DlcUgqQ8RTFiPvVtvRrpSnEuCLIfdwb+wdyhQBd23yA7AR5KPC1LXaM+yHbM3fkAutg6XiLotd9vitzpxgKEQOEZI6V+71e3VZeU0AOOkFIZkOOXg5ZUVOIcR58IOj+enkQ51JwZT7D1O8So6+AoVD4OZJfjAHvgHnZkogYAzHTHemz+5MSIQMvMQVcbjuYm0S56niDMFY5Ye4KnoE3CKPD5PvxDIXWeEZ7+AIHYh4KnhHGdoa189cqm0D1F87ObPgB7lRaoPcZeIgw6ZjKuVcyBU5UJsjG1NEth1XXfjqMTcHOSIIJi67n5NgbOn83yMiBOczQzAr3mTOQA7PB8Pu5g1UCM0cZzIEFGzPnEzSJsLEeGAXNrHA2KvOBBBt+gilRuCaSYK7EdeId6RTszNd4EXM/w8ICzJ3ByUbgJ7iWgPR6ZnK24RSdo1zBNbgSbJPNrHBzPBNZ0NwxbPgSkKjLLD46vhJDXRfQqM5kFoyGm9vDFRLR5Hv0AzT59zcocGA6O5+j7yDX1OclsE+cHRuh0xN92bvA+ld27IROI+WSc41VoXvUuRfqjLm7wPZJdmyG1pxPdhaIzb0z5oGYYyl8gvgUWHQy8ZOZBUIzSMT01zYjFK/ratMV7AwUuI6y7C3U2dKQ/0vSiFlXfyMC+RPkXTa+P2JSiDmI7+syB3GYAGmDQMxOgdJRY7GODpJAnAOFmAO6MeZQ0JVKUIjZmCNWR9AZCiQIIjH9lwb9Geg2BEbIE1RiDmj3YtBdoekFMjEv0J1An1f5pvAdFoJueDdsW1pAjycHrgdG4pGl049ZCobGrQivB++YpvAdFm5FhOFB23GG3wXCdBlbGg80jGuYwIGGLORDX8bu4UtAFz4LCdEnXkrZoLtCxKtkSyMhjquVwutEdfbp5Q+WLgVLDwaN0vBHa0qgYqli72gcPJrCC2FN/gztAfMaCoZmPkxfDx4toMWZIBtzujCG6s67CF9LATSraOPV8X4O0rwrh//tP35sjTYb52VlqC07P9PM//DNt7/77bff/NtH+/C3//g/4c4N9Sfc+2/6Z+N2wj1qTOyf1+eJrH8B3MF7xol9/+yft7H7BXAn4V6NcYctFXv3B9xFuFd8XgryBXAPWqJvnZqmnynq+wVA4+Z/LGjbHKdfTcXYk3HTvOD5+ecVjF8A90KlBbrbvH9eKrHbUBi1OXHOukaKhEBDbVSji9Mb9gwV7zZnDRHGPerz9je/AG7USI1I9O6WTcVPUBynT0eqrM8TXvgCsOE91HbZBbfnTpDJpeImA3EvdpOZXQQ37Tz1X5xA/uOQespksLT4dMY0HZUmvU0R//YHPXzSaT/PNiESvp315Sc8Oua5C5Mpgns+3B9A3KkSBqMxbEPYvVzE3MGSbEUCg202kSAYLGg8aNjJdq7VRTg+MTcpgyC47T16F6kYEu09gwa1dkr+mUci//9x01Dc9od09O/unzfE8AVwJ+IeD35iJuInHZNBw82uI6Ci0u6BabON22eB0OquGcYSYR0Y39q4IxwFYtdYIv4NQ1zHT2i+/Yi+DxE3Geje2eciN/GVIp/lwFonxzLEbV0kh6WTQ0cnkc6drB1OhshnORbjplMc65xKEsFd+FmOINo5gnkl4t90ymfj7oW4K1QasYU0P2lHPo/ezVDBjXGwqjB+f+7S+RfAXaTz3cbAsuHtFPR74qZLDHG2fpLLHRG6pGMMB3YRah9dpGiA4wYbti/asTrs7yJsCdcCNu4o6sTuKLNkcOPFgEz+KtdSSWLpakBcvQ/8KncUVLE3RpM5SRtp27ubSNNn4bRaX45HBsNDpOmz8MDLW1nZn11UHU3Fv1HbfjkfdVwqycnC1+1sg2/3mUjPx0jgfid9reNtx9lEeoNGo2Br2xXeLTfuGiLhxGgUbB2h+yD3thlDBHcgbjqwc26tlsjDjtEo2Cl+B956HGuo4O44LZMIe/9xVGA/nFMxvFM5a6iEQRyqWugmZwvHNFjeHqaT+KtsnzmW/gVg43RSmwtJfv9xVHBj9F6r4Zn68FUiuPG91Ujk83yWLtKqMsMxCGuOV8lzLJXPki9hGY1vjM/eB/gCuPnM8IAduUM7KjOxZovPI7OfxBB5KTZvbG9D3CYyUGDOw6WNspN5eRd52THn4dJZ7CZmIsm3B+MODCfb60WyKkfawdX3EwY/8xzEF8CNtBONXkjGNboKzfvgIeSFWWzvU8VPcGrQO0zFnqnBITL0bTw1mIZF2vKhAvthjq1h1TBWidBOGJub3dtkelWBtJMjMJw0lSgYyDo5HZPBmSIvaRYYBbPMuMUmMgdhwdFkwnzSqyhWYfnEJDaTi50ZKq3vTP4sae5kXKUSTZIvQy+YQt7JdzWRaUdLJPnuGQ/JoAjr8BRyj4lsaaOJhO+On6XRQ1pv186pVHCjf/fknub+v0HETzr7N07z7vCdKsV8Z/8uTE5cxr0Hundfi3PvEinRBrdOgqQg7GhfinjJwGRwWHIOu1TYcvB+68COz+ypYu7BwcSw1Fld5aN8iIGBa4utiYxO2+DUu1PJcDZ2RKZOrDCa7MISuXJn5CIlWj3kVB33FqumyGc5sZIfzo3B6SJDVTYfovdk3EulwTbrgXUwnlhrIvFkLsY9FuF26yK4F/t39wfaEVmRssX7xAPDyba2hrkdhu9GXmlAOzGu0XKJ4HbCXQOkk6K2m4h0fJz1zEbAdFLMHb5VzI1yZhU0LXOmOkQWSJxn2HzCTGwcoeahYm+UM4sJbLk/y5mmYm8cvnNoQLy8e6p8lTx7Z93ITcyaSvRePMTWMHrXLBF788zgqAafZT+vxy6C27hoAI2W7SetUgU3JlWd7tft77KpdNicJc16WxhPTGWTzo2LhpZIO2uKDLE5D9+N1tHeMg1k5+G7QdMyx09yidAlD98N0nzauN1FFlydp+86yetv3FkpEk94im2QtOPGHZEifMnTYKOgqXlwD5FwwsNgu0Yb6N4h8tLgPFQ1piPuHlPFvRdrsU0sLnd4FEljeaqqZkf3zi4iv+GBn+X0wvDdTeQF0Hk8aVLv++BWkdJ0Hk+a4Ri+d3QUiSeJ/r3awO8yTWT3xXleZpKU5innW2n07J3nTuYqjN/15mjUu+LGeLKwGbvtPUVeuL3jd7mS/Xv7ich3yQMci9QdY13NZHBzPKH5pI3behPx7xEsmU3d2HPbVqXtM1CuqtlMxO0iSzs+inHDxMzxk6nClyxr1ugi4cHdTCSvYlkzs45ttuUqvFPoJxZUXq4zvyni34UyiUb3FA/Pu4iAkk9Wi63FPK9y1MgnyjvuMgjbg1Uiiie+UN7R6ZTsid8lspvrK9negXnsLJF6PkAZ5+Cuhv69loifREPe8VUP+bfIXkM0zKvC+HmnxuwiuNm/6ymPXSq4UVY4YPf8hJMUaQ9GwwuiA69xVp6leRX/5jOiA7QVdo12jd5UHIVPiS5QSrSK/cWK3JkIw/PaR7ISgbeSAQ6ibOMqKtWs7JqRIrgdcXeYtj8GV1nkDnih37jTYDDs2NunyOhMgB5RP/om5OCvXpZISCE9onPIFQ6Lng3dpuLfQJl1LYOW7EuNSAR1EepJ12fPc7EKX9IYxzZ2g3fXdaWLTIUFTXHUEdQM8JHhJZJWwRBHHdljeJWqc1S0RGDfmbLWLmruvm12pUgrNkA/qTbfvIlzn1D7/sMMkZfigMGT6pfBjcs4jbdSsfadJHdhUP3+UOzzpLUqX2QnJ1njLvR9kloTaQsGjMv0fvTgUcd+vMkN3xU3Dn2PBbrquUOJyE5D+GRd3hYEuzeVotJR57toW97ObLII7ECZ7zkWmnumSjAJFlx1Q+8u7yr2DsYNVzE27lEi56CDD3MaPTKk7+ioYm4W7Bsk32z75yKX9CLwVs3+de7uvTOqmjLhBEuzyKDGwzVleseBtVmbQY2eo6mk8llicfZWa+gPfnJmlkRyE1IGO7DtHgXdd9HWVXAbusmA0dKNu3WRvdxIZ/eG6H3svUoFdyDuMLT3TgpEshNSkNu4HS6keZ7qXsXeUKId7SS7ZbGHLVUOc0YOhN3uEz4vklfJYWGQt44qbxHsnk0F9iTYrd2fz3Ym6Cpj6kFyfeuMZ0yA3UTO1AQNTe+Eb9x7PWdXp6sE7m4Ee7w5Gfoj7LP4L7KJFt0J9q7Wb+lUttN9FckC6a78uXll5CSriyyiBZ2VP9a+90xe63NdxUk6wfZ1V22OcXVXiX990BPlTvUKYC8XUR4KUs+cm9uDPsldT4q8wHcsKB0Ekza3e8rAXsTtEw5hnIvbpjLvMPDW7E4A4UZuXZtURUIJrYzkZfT+3ncsKZEbuTGg+5rXCtLV8mu6yKh0wKrLxl3RsWvso1TsjccKe1/QNT63oVUeRQYKPQVdZt+4s6k8QoHK6hHWMro1u+OMyfg33vbtpEhwbhWryJUGyKzueLfmRNztzdmxd8WNk6SnLUvxu1JkhSGKr0MSX1qts9svQpiF131X90TgfYqIV0Q9vFeCdNzL4iESCQvfK6c1xr0zRRVPQcr0AdcwrF49ZRXgzJkTckKrHdxTZO8vCtUR28yOFl8eKsBxdKORWPYL+FShzWlE93MNcpXNp02FNydGw+a4OjKvViKeMlFwdb05avAz3NZVxk4mBpXWJgOPJqIBGnM8fJuFwHdmKEKcs9hVki2eKrcLA9aJD/CRhsBnU5n1WRgNl3cGvmQe1EBcvb9Upg1zLJmzorFYf3pNzsZzqqwUL67v50M23pdKcrhYE6KvicBbD5WPE8WIdgYzHpJDEVHNWKjeZ7NxVtuaiNptNj4hvjw4V7EvNYt3/vLPr//uq19/9/tvX28MP4L68ff9Yf/onz7927/8/X/19W++/v6Hr3/34c8+/cL20ed/Nd//4ycsv/xP3iD+w3930P3XC9afv3D985/+yQb230tmAqo2AAIA"""

_ICE_MONTHS_B64 = {
    "month01": """...""",
    "month02": """...""",
    "month03": """...""",
    "month04": """...""",
    "month05": """...""",
    "month06": """...""",
    "month07": """...""",
    "month08": """H4sIABs88GkC/519S89lOXLcX2n0WnV1+Ca9EyTA0EYGbEAbwxgMZhr2wK1uo6dHG0H/3eRXt+pjBG/yxOWuH5V17jkkM4OZkZH/8eNff/3bb3/66cf/8sOP//Lf/uEf/v5f/sc//9M//vCP//Tff/ivl/eX/+Ff049/98OP/++3X//9L3/+6Tf8c+P//PmPv39Y9z8br+o+/vS//frL7/+n/8fa//l///aXP//ht5/++uvPf/v9L7/+8of/+2/9f6Sr/58//fTzz3/4069/++X3/l9cy+H53/7a//V//sePP//6S/+HLy6WxxVc/38//3H8wdweNebxR3/95U8//fL7b38cf23/H9cj/+ff/fBpV/wjXPG7Xf/X2l6aJQ92MT289/PzWn5tGPGBMT+m31mu/te417+zol15RJ9nw5hfv2Aiw/pIZbYrH+/74ofSG/Yvk+AN6/X6l7aEhuERWpifeJXXhimiYezv5GfDENJrQ16LlOGbpo/nv/imy1rkj+X+blhjUB5Y+h+sn3bucdXXaxEcGrr+zrCILr1ejND4m+YC3zQ3Y7s1/qalwjdt2dhvgT9q82F+R++K9MR+EluaDWN+vYqhgKFv/ZB9PjH3g9L86w3HG7Um+KVXatIOD49WwDD416+YMn8b5+EVU62vnxh2h9/1bfT6DUPgb5p8m73UZTww8CcN+dMu96VxrzdqQacR+s683GwY0usHtsCGNaXZMLfXT2y4isH1HfjpiHN9+vMXhrgYwfevnGfDkI1tg+4mhIfP8MRs+H5ycCE+nGuzZ7zc6ycWesf0uGaz8NrZRMf7+7pg10SXJG8THg62t/V+ixv2Dexaer2EvvGxiM3Nu9RbPxR3aXmUkueD34zYTQvh62Nyw32v+ZaUle/Hwl8FNqlxDltlw5bgia0maZP2vRA8bFL/yvF338WnIqHdy6C/mI3YXWazZkREPoX9TACKChY68RzYnMvzninZcFALyvCz8+57Pb72wWFBGdGX2TC8XkKfeXfnUGe7ZCyhX1BGrRmQorFpPMW1Dk+mJ/btHZtyDn1+uAbxsBpxLS2GFXa3dQz5OPUog9Ei5SaeQw92Rtyunk/ThZ7bG5umODasFxhmFxQs3I9hnsNaPxcG/iqJz2H0cT4YvjkpkPb9XcAw+yjFiu68EX63FBQYFdoTjX6CDG9scPL7PeY3uAo5AyyE5fAHsIuXhmr62fNgaPmMwke/RDiJLb1eDF/5CF/XdPaHs3u9GD7wEW7Rzzs8liZcFPpJBFDT7w3B2OF8EkNs8z51hv+OhU/iVcEwGaGUFj9ccPfq2y3VKG23vvgXgKGv14YXrxj4ZLga4SacDM9f+WRcDa97tSgePOS+/AjbY1OATei4OYKh5Ylpw4W+cFOWYBypHARE1M+wD7DDo3FrY6w4vhZcFEryUjgddxg/H42vrvkWLPYw6NJsFwxX7BdEFBwuv+VtKm84V+EOnSx4umy4i277hg8nHN03XGvSvonVxvtj1xTFnY5tGnHTaNmFvk2dB/wdjbvXuk3RD1dtj4bWYMe0IuUyRn4GDIN1ey68R7/mZ74bZiOxhL80dagYpvVLZgwmh5HyM+f1NBz+oygOI3W/OwXvbuiqAhXDyF3ATnNNtGsT5Bs71MhkxOVMtCR5ttUJzzeh3SbdnYmOAK4qbdH+7Svs7WqtYeVD4S/YNC5EyUGNREKeDaMRu2Pig5EmyNcNi5XGXABYnVO88XF5KXOWxg0XNni1cA16jNS/f4N9evmm+KgUZ8Tf7byWy+gn6ppvX/kJq188EL9NGtApzobZAJm0/qkj7lJmQ+tmuiKiNgcoG0ezGx6OvkpB3/NhrKUp/puA9DiMrimHgw3hMPajYd29+Sw6D0cqGi/Ibqo8r02fJ6MEJWT0s5jmKk0/GaUoIaOfxZLBMJSouKnknhn2z41q5Bf4LFKwyWZuiU5/GhUlMKxBuWP0k3Hlabv1fzUAOJ2M3Df4lAUrI8frlGXMo0gGhtG48JO7yfHRpmxWN8xWfJvf0Y3iQ4qzXTMOP7mbkXbLUcHRjBVHAuttRNTR9vxd7FBD4Luf31qKdKDa9vyWmkWE6aZETz8V7tJyC32PVIhQ0Qg0jDPqI8+AKD6KVU2qfIDnT9rf17hekBNOI4Xj8DQ54T4zIuIcg8fN7/Xap8QRMaQ2G3rjNGWOiH7+of0wGf4iMsR0rSlniRa/H3t/wQNLVVB0HpnaBIfQS+maHJ/Fo6dhtetzuLtzRq/f39hy3rj44+TNNe/at6kUSPv+giJ7fSZFbnypGx9xSoIMuyqUIN94XgQz+f0y/swBaWazYKQUId/qRn4VXs4XIfHtvH/kDMnW+jIXwcWLUQ6KRSkJuMrJr5yVPOSShs7OKTkTv80m727Aflfr3FwtCAWPCHM1CUDlJcLMleeNx99FmPisudyyXHqE8Q4iTIlRADNhYAlw98m4OhWOL3UG68msVxPOoxrp+NnG7YBy3u35Rp/u3voyjoFlSgiekuIMGVeW5+G6JXJ0XBkKRLRqfBr6pmkEe3Db0Slr329cMYLb9krkzf3D5ATeyfCF+Lg8qgHgC5v0ejk9WgbDbCw9obWRccQPUw1GlWeqWS3wglY+cAlKkAyu3YlLOZMPTABvaNXV5we6es1HvprV/5AoBIoPS4FCoPh2BWOZ+jWrpxAoLh5kdd4IShXtuqufSk6DK9LueTTODfcCNJpmXAaJ0uY7ZipBKYsvHIyIN54sMZoGBcNVpW6QOOhGjJ1mEngtVPmsZNfWQtXXdMp9DmFJy8DO3lx6lhJuqU4KgmUTdPu/GGX47JagO699el7WbkNEj7r5AsNsAZklR9pmRNldapMqanGsK4TdFkUCZUsQr1NpEnsj9HOP98Gk0j4chk9nhDN+xf6IChe0UJJUH6nPusbtzS4uaWCop49Ir9yx+503RrhOXMbhZ2gRHxTojYtB4Stvupxy68l85c0eLiLZ8DU58pW3enhgMYhiNTG2uIqH4BReH4ycGFyEOZXb3b6B1viyXJ7+7Lth/0/KIg4u8XynaM/b+p0jHiWYDL80GLlj2m3DLwWwK1Kyq2/LOd8x4oCQeyJU0q2MPBD4YP1hwPDV360yJhG/ZS0ESsS1A9rdO5sFnvfG7oT3k8FFYTBTG5BSmpEXzY3s5nLBhswCIMi5DDeJkUYyMlw1MHiCPLNNLIoMnlyVSBBEJO/oCRI5dkU7LTkLV6JSKuSCyIVw2y6IpIUBF+aeBzuDmxzDp+SkK/1SYS6+SGgmbdFTcU4yy/3iJdVC45IULzMiHX9Pk5LiDbeNXSrixoXr+VLfDZuFgfPS03MBmPEWtEjbVgm75FMYPRUHdsVY/bJteOiQ5FLauWL38BjpvZFeIYiQ+vbGBG6RXAanZUYTQbgPFR9pma+8pc+LaFUIpalgYcP23W3J+1+uQJDxWWnpyINMrgTD1DjvP99hu2Gygm9iDOTmm1p73r9uL7+DXdvgieWSeuRGqwK+YylVQl0RmMjdsEpp1XEbaLAY1SL4pR0Kqla4cISCxKchClJfjtCM/DGB8vzG6mG417cLgiB5eyIqkY8Drpx+boGHovsJyDO/4Zew1C77wYjIMD2ztbddFXTWfXy05iVKdloo2ZA/SlViIGFzRLdrRh5v6Y2AdGPHTZdkV59l0u922SjtU0dNB2qYkrEK3yls6bF2JYSTTh6dhJ0EyoWBWs5VAU5boNZhU5K2zABqDRhduQSttlQzALxmbBkC28x07ajJ4MZTrjK6DlvAMJWsZPI6TstNqtnUsm2hrg8XlI4aakytTxb5bV9MHADLKSkZhj8XsgmM9D3XhtNwvnBTfplOX8xiPyaIfZrUEsUM4A4vX+6Z5YEd2keMScbmrnlbBRudTtJK9HAG1OHWHZuRe2CSxVygGAAjCwB9dEBNDX91dLJn5cr7jQb/3c4bOQRON/UlhOd54zYYlyLY3IBVR0pZUHhApDWsFJ/2xtMqIS3t5SjfpH7LQjhLXLoCSETeKVhxe2NjIutEPgkQyN44egERk3jUuYCp+xbcKqIrozv8YNXE4hQpALjhjkJkwS5yC12XTEm4NLOZB281CWjeDeAjSV1QsrDHngtakIuwOTvQCkBSGhTKppB/IjLgw6NGqT2Aunr7CTab3dcKrQdqjJWdjHHXntsNrYOU0o4XlZ4f+LZ+SW22AxEafLG8YFDAPXZZcO0JnN3nJpW2QtArKJDwBoJavzMv9KaYsbZnJEPzUjC9MB+WDdpQWXKFOUjsSwIU1JxbTYJT2RKjbFoNI0nqsa39/BYJSvavX8BvWzx2SsFFbCMfDjhLogxpXJWgDBItlYS6TfrZmYq2JP3KXE9sJpG9lR3bd8Te6qUK5lgggAiGwlHjwufshQeOMa48nGVEVlU3TBZKQDvgmNbBrMxKm/yHvBAaViXBPPx1Bbti5N7Jtw0CQAAUVIpw9Bc8WYyoXQPhSfFpjVN34uuRnfw50U5evhapECnuFqRH6duz4s9UT0MBCKufPsy76scdsZruX8hOdmjI3dM9KFbXdZeNdWs9RmDd2rtn9Jrqz/dkswFhuavWUBrAG8iAoihkV6zc3aL4AvX82EFlEbGhm1slktkUT0XWjg09tjhbST/P0DBkSPoZPKy8CCGlrPC3ytL1D3IBdsmzLMx3h42jVvJu1VEB4pcN1WrcQMMhERWVgNuhYYrQjteykjEaUhFV4VK1pUFqJleUofbohYDbcWGBxJ1VEmI82aG2gzPvjN6FugDD7BUqTlm074qPyn2+MSxsAe7z1ZITDJxjnHOaIz9ilCRIPCvF/tOk5A8b9iifsxQGF6p9maoSI+4W6afmkUWAn1q99HF6xPaYqhKZdH17T5SFOoQOsnKfGHKOEw2kGzrrSpi5CWzmpg5D43JHiDKDlkK38y/JFXiACd91o3ifzz591huvVgjdid8S7fTFa43gnbZZIOv+zuaMlPlTT4Oj0qx4/LAvTj/vlRJ/on/BjJru0ODK8oYDxebEDptcK4qOQSLUBDx7WwoQUJNz5ala8KnGapSeKdEYno3ytyxBlp/DjGHHbJekKOIj8hSSKQZGtK0OC0uCJrVq0AtT3tWs7Sa13BY1sFgUtnxdBPbm6+OGYFbDtiXSLpXWtFWwsRNjddG+iMnf4y128wjvngKGtyBt6LkBUz5L9UfWkrGanNiOJfmacR9Y7DwKFNvcFkrfxNgPuZe4O2VTQB7B0IosfqNOvql8ta3kzfDZBqznii7erUdm5KUwwFLRReWaEcu8VLHul9uppXXEXGsp0q4SPLBBktBkjtDKVd3zFnsTBj94d3NpqRumS7rVUVPdMLTYoYkxWoZfakUmSk6P9h0PdsYb1rzDaEVB2erDVoymvRpimDc+ZSMSnLh2lYqz4l6hRJq4OemjHB4G+fBx96V62Ony/4Z3waZU2ZtlzqNpzpOzPh6Zj8lW8AkMfVwB7FMs/a7I2GfmDgxhO0P6PC/YJyBlPVaFZ+Kp0mb37dVFQjW3qFDd25ISg55Gm7TVFrGhkKUSXVuwT/X3qZ8FVIT+vHAARgj5vG5pewF9CuWIU9KgT8IcccpRgz5pJk7anGVKiA1V4QZn11n3gbwblVEHdVSCFDTxYnfvTJyiSgEukM1S6vaconJTvVu9yH+kqEoDBxyishqpzMm7EV0MOlRbNMZSVgI1240s42xXg7LZct+kCV6vJSmLOmSLJxpI9U/B3xebpjGCmcgc1T8TJbfZ5YLK0MMwCZCJIMy4kichnulPwwSO/naOGGbi1yxEMBNXr3A1UNksdIrUvckQRjsJbCWeO/6N+kGvnCsSPQuunO7KcFvqvpOahPyg5DgsXFVJ7AmFNUblqkiDI4YslQNEYenMpF26KJtiDlwMjJjXsgUEygJ95mztpueOfNJQj7+iwh5vYUcUs7hCS4jHYUFmCerFcJOM3CQtK0KDe3QokrB56rXC/QvgA8O+rCL+C+AzS1KNy4DTcE+7pGR03RG2NgyT5ndTezbJ/br0aM6y6MOnGS0xNH4nXc+q76cPjcoX5dtcv8AbdXyur+VnLvkmsLxg8c9KVvVbVu32DXP3MRMHuX6jNSrlNSDE+GfnyX19rc1iXXVIBxcl00sySnUoGFTlgd+lm78bZie4tQXAZC+lReSnYeJAfz2MZ/LnzGXHZ7KXr3DFS9ovjCrE/clmR8fhneNXKJkiHXcmeen+BftPdYdG5RLmCW1k1D0btuqxQKMou/kLJehsUXOS2PMeJ5nkZ8ritrFsJH6cU0SJVhTjsT5jNvLvefJ2R1rZ1bxs/nHbjKIyaS2L5mjjVmet0EJzoeTsBk130kFFQhbp64afFxCmYWFcw0s4SWrU4ZNEnULO+bi0JJVbNA926oZRq1ph/lS7yS3E8eEHg9RN6KCnd3MLb4tG9FfW2md4MErcTEkiAmoPY0n6Mpyn949i5Gu5D/Gb6tx3Q6svhuqxxAT2PZBXBb5+yPVOvzT0y08Ugcjc8DwM8z3Ph3BIN6pJ4uuIz2JdKPHdStmwqjff0hMKUdeu7Yo69mZpdUePtnbnmhORTgOb6aevbUjO5mlnCCk6Fy49yb4MrHSXS4UZV/oWk5rwKY9C/Xa7QRcL6PGxKQr7zGa5gBW2kU7OS8oneEnyniik3DNpqwnVXcpn0xWft6IWmx6oxqAn1KiQa1vYylNYhI8XuCdih4KGl0hlQk1ujNF9/gCIEOqpUpYIp6HtSJJpJ/ewKaU3t5tPNi5ITZpjTKoNI19u4YJl1rpHj221qTMSQfkF8158l4LpIaKeCCmMWFaUB+ahgQ8R12nsPmo774a+ShmRobWMwdq6wze3UzYIJi2QkkUjdT0nCsOz8HkX0BjBvO7LZgSjP8xvpAbsl8O7+BsfcycasFk9aiiTt8tCMDnYnvJxYLOj0yefdjbTvQt2acnuDLu7dP9Jismk27lpDiJG7gA/JSpDeygdMrjKMQjDGCpDn8gjFaTCzPUk4N6qE1LZfog3JEn5uDGAuTA9YfZUL+meUO+bdRY4kZ4Djm/ooy9GvsAFyeJsvIAvVxP6R19UrBrWZl7D7MUOJ/TtMMVm0N6OPhF2A/NGadtoJFy7wkrWCv7LTO4YwBFGDfsMVQnh7rjkYEaPJTgnrV2O5vMNR98kWi2N2TMj0gseb5u9aHjKotweJZqWV4esbZUyKQWSvDU865C30IfG5XVDk73fGInM7P1hWAUGFCGR+PzZ90hEehibiS9HZurHZDNx8Vaeq7hZCIaIm5OhoHoaGBkcHb7Ds666FrJ6w5MBk0V3nUgmUj01ijnpgYHkGl2EnrXN1E/Sv3T9+ESnzBssW5Blz5GqC8hKKdxPiWDH6erMzjIVRRcznICjCgRySc2SMljsRslVUA1b7AIJX1xSCcEnUDIw+VKLXSbFtyLlQpDYYxZkXsAkcNGbTEjdzanb4ZaNHtM460VqecGxcTv2Q931Vw1P1rS2JSTbaOWDpVAVDL7+i0LVPGxuhJMWD7I1Pei5oLF9Z/aDFdAX/MAFhGjOo6hb1ks0JfQIInG3TDQztG07U62OQTb53Sbvca/WgM7Js9R3Y8Qifks2E5duZcNqO4UoIUcb8/AcnJ26syP+jkdpBFc0D4alN9lfOkIronvGeqQeD7hW5Hnwt5WH4OpUgIaxzXTkzLAqe4kCyygnPVBtXevFYVRlya0vdgWIX6qesRvcUncv9PQCHV04oOi1hMBi96xe3vS1vQBV7RLY5Itd7BurSXoT2wE+VtH1BTyax/CYmc4X+Ag8++ZukjYzcXanvW3hUTUuJyxlhBNqdkWUuBs0M9L+ytTLZV7MJunBeSQc/LJLQewGuIywZ3DwuCMK5ytUe8RxS7t5Kt0wXwqYo7kTdfyrRAKi+Rg1PtPzdw+kMR7drl3pfazTvAJazp519mryp2QzceUWfRlxq6yEFWVnrsUe6SCsqEU9eJVwi3jSMWOluxZEgKonI7gj+81AuEX00yQ+5Rz0nY/enSZpxHTDekkzPLki5R84v8Eauk3eqAOliDJwr2dqLkgi8rBuKaHQYVLM4V70/A4nWcqWL3AStAZamkkvskhQR97IhHhGShE1oV430L2APJBntkjrN3mkTaNs2tGrzeLui4pbQo6g1AdMPOnd9SbtlL11CIKOwuYdEH2chLaHH2yK3gBRl+Ozvnivf4OSpuO2XjSRwgtzA9mALXUvY2NFzFsq8Zh0qjgKltxN/aqmKUIjiTV1T5E11AIKxmPAVHwXtYyZkUVCLSfPOnoz+UOuJaWDdXtnn5QdW9bemMSW1c4Boxb11BEckA85AjnVp6AOkO7C8FO+4TNx0ofmornXX40IizzdNZIPQIax1GTCYlhwPlapUjeSu3i2nZduQs5BA5tJh1nQgEfK8mZ2ZmB0NR8+eQyUG2D/oHe4wyugsVnKkjfwaqN3xPCxYJ3b0hl4ga9Cqm/39i11uh0bvzHAKuGE8kqyjbvuhrCruam1HtJQ3OVO6k4KcQdd8gYqRbPRLu20Cc3yxAJdUGNwk8fgLi9Mx4/Jhk4h0rAoVzIr3mv7N6iH9btLKoq8MhdFhoK11nZFamypfxuJt0WqcclQvr0BPRK15exR6pvdlJesL7l2Kksrt/Jr1Z0SNoUie2dSZ5F4EpjZoh+8ZKMX85hTCkR2KminezGy05wmv9sbTpqGoolBAck+ehSiuQlXAV2ejTYLjQ3rhkBrsUXiaBjXVXmgryVbQ0X5js4cko6LlKq5GrTnmfPCF0gw7qrxfSYNpcw2Q1Yo899RXQ3aEJnCqA5Yeqqka0dnHsugmogao7ONgELgct9VvNTymHckqo38BX1TUleSabmUxlIJLiR1tOPI7kbb7dI1bSc9tPG923TUN9Wr+/QXiQH1v8VwvzzcDFkIVkJjWUAWzxhswZMGrw4LiiQezFokfUWTpH7JhJXcr1qKv2Bpl/xQmLkIlXq8F+rth486ezH5Q94keqyFW8tMB/vkjW1ZCPKIx2BHc7FOHeMr+YxT3kX1KdgupfswstNcJnN4VA/NHXl6SFjmuYoxCOfe6UGPlAGv9CQY35bQfGTDWW9o1NAM4XTqob/6uc7SyMvEdrMigSyp27Eg1sKCxI5hJGgNRn4B6Hy7H373As95f0Dice5567rR+n6RpCMBdC2/09Fc9u19kTeXkA2gyVCsaE7r/VwEF3adS3Gnm2CyH5eEEskfbDg1ex0Du6hV/E6NYJcaaltZgU182PXXpX5ApE/KrapaNmNt2B9pCnfSd28giTX3Rf3Jtnw6oytqpM7PrpFbdIX93kP8u7yProrW0HXwpLP3kr/jTcO2BgDVXbJ2PB1sSv0IVAJJ0omjh+knnGphqkdpm5n3GweGWOcNj5k3M+g3LhpLb2pIYPlHNQSRXrYa8UbXOSIBjyPo7SlNNDeiG863hVHp89LoqyvgULD6cIYgKqPAiKNpRzNZFVpKrm8F65tS3wKSEpz03eSkzIZzznmU+qSkUAePGWcLB0mW+Br8wHw/JO8GPe7GClWGjzEeSPlTkXcnolu2CcGN6F3dVWtlrRnqxZRbw/nYW90sC9zB1sgNZ5nHs13c45NFw0CMFbPAsRPj3F3327aBUMVXGMuy2l1HvTe5r5+mv4R1n34iizJCjlMuWSI7r2XF0t9PwmTUEVa0Lm8EZWNQtBKp1Yctk+CPXu7wW6ord9Oktdko2YZY2raUz8CShtLOHAMz+YS3XSFsQ/fiEemiB0s7oLTxmFCvUz30MsldjAgMsOQIxKjlwvmL2ZxX5hfD2CR9ZuZgjVs/ShIZEqOkxni5+QRt5o7TjfT6Ntf1s6wowQ8PGlZD4TAXza5eWFWUCOBXQImL0QAnwTnEnbuiYmBD6FHVZh5/4MfkozS1sTKADE2aflMYQIYU39WcXwGkqvRzNZyWHESNJ+rBMLVUXuQRs39XlWZpiNj1g7ktCtwwbbctChZZ5a7TQK3wMT9Uy0qs1H8raN5huSLlxokRXMwJUlz3JN5ykSp8Ky2+366kfg/igI923vK2dmOxNYXbhnC+eZjfUce1dzv8lOrKbbNXYhLwbEvKJ4BzZWcn7vCAv+NPImEk0YHh5hL95QLlRPfMo1LVcNADL9qJ4acDKYiTbUgABJiFbqipYTmx1bnS/TFpowlIrptFGt71+mvyw2bR8MEOMwaCIB+tPXUDPrlhSchXDrOI3Yb1EiBAGyrRB6NAO0addcPk5r9r7B2AVC5Jdo7G6uZLNKtVUx9Iu7SqKY/xAqP66JQ87oo1L+xrDVo3XjdsKMVmTQjYZzlNXd4X2UqYY6jp6K1gM6rXBSQv7lq88zbpaLVZLSiOuIQaU3lFjRptZ2X2SRWpJQGo5UsWmp0W0xn6FVucaTvJrojjXDiF1J2n1BXEPK3afamSoEEMNyREFJqQ+rSb/Jj1cgul/eRbigvHsOpkk5ztyLPtf3jYDs/2O64kETJSfdcOGG3E1/BxomsmdogcChjiJJhvuZl5hsz5NkbNAzSyojIBsTxfR8Y09+u1EwpsliooVln0ush2s2j6IJG5fC8gOuApZAwNuUsuIXWzWVN1cMi8guBKX8z0tk43YmF1/Fi3SoTDlHJONwMlE7ElkTDtAGGSxFUHmRfyKbUYd10PFNePrbyPTe3RTm0p3cOAZrVGSZV7Uf94hZhRFPqgOrqucsT18I3OQNxCRa3FbK1Pb4jr+zKzxiVaIZ9aNuQajZWPWVtmsZKk4ikEAUOJxkszabAk1++iQZpfzIXD7jUs0mDcwbDsDuQurYfty43Wu23RlPUhGRedLJq8R26AkVh9PTsB75y4skM4YhPpGx4lEcBRXRiYiR6T97HqoZk8pkaEQSyCQBmgF+NjtGFRgFhAopM9pQUce4sw2W7gN6PmHgrZzdpdA8AZ/RGRzALqYVkj33C0brdLJd7raPWoz683y9Sbs+yZktPtJt0FsweAg09H0TODYfQACC69W2FpUpLx7FYgDdc3oDKuvSPh6xIk8VcznB2bk1LO7GYF8JRzTsLB0LktjkHrZqlJ8zwx70b4WZ0QRllhceTFgoQ1AeklTSvrTBKkVYWeOGsq8mNWaKo2DvLFVaXEc/hRaVucB1BLixyU1XwWZVRsAEYFbIYp2Qpbfpt2G6NPhPk2hNva4yoHaTD7YTvmlv1u29Y8Ee6drdvhNjnclYeHQD5zC5YSDzk/TnMpvNqqB6P1Vh0mlTJV90ziE3IwiAg03BMXPiN//5JGgKQ8mOuYHsb1Gky9kNkM59Nko+2T0OW3/MmncqpXVMiah6mtA+4VA7c5svMXTqcx+l/oc3oYEzTwnrHmOZNdqNCX0KLS0N/8AxXIpMF63Spf4e0xvd2s+PdlxLpZRV2MIvjmcTEANT7XgXuSzFyV+go8mYF6zkZ7PZJdKmCXnZKYouuSOfqd3w6vPea81dUstLfVn/kWsmt5L5vrhCjbvdwMNN1MhviW8hQDPgbrWsmT3KzIP+NQoFHtFwy8YWiVLZbd1CD3Wc9N1mw7B2wH23YwsV9Yo0CPXNBeClUZVCI+C1KJb7waPk39ljeYaLN2dQNurL2CTzvbmEeHQDxv9Fri4aZHiY6EXILqthjnyV4SKqWqT8YmAjUALIXE+cCMsQoGuyA422rMaTQ4bmVnVYxSZ7CN8nOe3V2jAltZnVaEJsFq7EsJKpNVMppxdx/eFuBPbmdlSoeUnVVLUrQGq+4QlKFubFQVkQswGvDRt/eNpBGFaNSho9InyEYSnRCNLBmbG6Ps3zYSdWPZKMW3jTZT2NLOKuaDRyljYchIVItarKpUIm+SwtedVXl7+1mNlTdG5m1n8/Nsrhnxe6XSKLXONykvt/VkMr4DVPj8K25stAcRvDhINbaDsrK6SgiYpA2xe5C28w52eDsQy2sHWsRH7kFzRHSN11wesaxU7wp1Wc2PXx30z8do5JhdmnFfMBRL2MzP1cdoDnREhFSfivsTXlSSgh9b2ys5yMCPu+Zkm5mDxNTlMPNaDhLi28i8Jy0HGcluHua9y0FWsitzMdfOQebCdti+YeUgG//O7OJ9DpIi0Edl4e0U5KCSYKVaSkEO51Ti2ynIca8N9e0U5CCu1HKfguRkYrerXspBRjIDhUkrB7n+yujjfQpyNXN4k5QykN2fNkxcWhdQTHgOes0liS4hVKlH446/1KPJgcPsYHTPl6qJ3y/vpgnBLmaaxBlT6eqR3seXIneJwnoXuQ0CC8BF5u1B1exLlku5qN0c1ZzgvAQHoFF7Dtqo74RW5YiMWI4aScTNgUZnG7GeCPvVE6nlejLKQvUcdJhVP0XHS/WKSCbUfTBkKnWXj4KfH2v0GeL706QaMYXBAVoNEhvWbLtdmqeC9y9rYPhYNvGzHwQro8pmYYZZHbUa8Cy5TdzdECWX58193zuiZN0E7A1TMnLABgqizZTMu0C/oUpW/p2R9JKVkaN0Ekym5GqGFXCTKJk3sMIiSjK/spul/DZTkl2KxZRc8EE/ac3dUyU5gUd+b8OVjGTm29tcSfbMIldydMRcAleSzRzNj3nNjlmtPIoKSExJjm8iU7KcDeQuRwMxy9HgpQXVbQhbsCWLqK/L562I8m984LKoUcJHIMt9sTiFLMldKs4GnxseIVh5vSDtbdCqZC3VJ+2grvVWaCN/QHilfNSjnY8EbMqREmE5EpYuR7M+zs606D/oZKrOinGr6BoJt4p+mG5potPnm+QAIjNA65HJuF8Tbh2QP812FrqOaYdgxp6uSn4wPdwMtMJTZ+SuIj3uGNOQzYGvDWm/uIM+sV+kvSAJOBrOI8JrQxyEzWKBpHA28gfY+UygacDr2IQpA98Hs9w2kmcCP7VGpY88VxtrbfrIK0O06Op9H/man3Kp3LeR7zGa3UVOINnDpGGzi/xFOswHoY18SS3iORf7yLtVzO3tPvKh3+PfH23Ljk/sIx+6382/3UY+Pnm97yJnJE+hwOoiX6Fku97vIR9H5Xq/hZxDo9hBPtxVfHuo5pdyNF19ga1i83jWh57BwcmH4zyyqJLMz0uilN9ip6nHsD+JYuc4alkEuW8JFQOOWJsnTeNHDwonPUTxoGM8nSj9pCONx3wk1p2PZq/ko3F4Z+f5zHmonoquQqpfpGOiemHK1qk+H8lAYnzpoZIhVgBYbWkf+g3CCjb1gbNZcz9UMKXsUraTYGV8fQPkOjK7KtgVCxz7HTJLZh4mMzTL2O6VjbpBcZu0W0fHZjUvkJ2LRZHJxBQHQrqNTOZilkq4lcncp/lMmcwV0mGrV5DmiRASNFUyF5R7dYcc35bJpIu6qpI5/nqX3xbJpBMuamSO+T3O3Utk3uQwRYHMvptKkuYc4cKxh/Uizk1PYsWNOOYKWCFxJEpjjtmyrbytjMkBThTGLBdOmrK05RjnUgLOkrJjs4zawKIoZtZnECM6PpnSxplgURIznahcEzDWdBW/BE0JiDlNXmxDZyVTsZtpC6dfE19ZTOnkQf6ouT6ciClFVb10/n5H++JoCx7t9nw0JfrsHKtOg+Km6qIYTosOkVLOqvul+pHq7Kl+pIYWzlkiuOq/sSrSSdUxL8NiQyYGZaGCXTaK5omf5xwQLJyx4JQBHiAUMK6RRKEEMOGyMX+7CaW/kW90YBcV8cuPnilhps9qlq52P9FnNXNIAqlK2RzhnDnNZ7Xywb87y2dMfwzvj/IZNOyW357k85Hfr8ogny2r0Zrjs2JAjwQQMWcZS35/hk8/0dm9P8Kn+Odl8GaCz5I1dsuYUmWCT3fMAXVOrIkN+LhcUT5ZHOCTywMLj9L8npxQW9CafnEDOUUqR75QRUec3ZMKSv2Jo3vS0TzhL0mcdrfFqqJuQjgSig+6sikEDi9KJi2qCSed9wdtWkfP8Uc6UG98wjmIHq3Xyc4424RnO/7seKlnmTah6jkoMKl+ivkzmlMkkKt6YEarssMHGrEaXjgjizXQAZUNNiOyKsZo53lmTv+wxn0B4SNBpaGHJrU7jA496GETYzwm4LtdDe3tlJk5qntfqzUHda8gK13xfkz3vsRrDulezUJpb8/opnqLOaJ7n59TB3RzTUicz13GyPr49nhuPjvWdO4VmqV83792U722JnPfJAOtqahrMpCGqVozWDGP1T2l9++zFdgvWyNm2czBNE1roC1DrMaaz1kyyyRveClEbCr0aoOBv8SG6jUifzieTOIOZ6Miw9FwIi9qly/6/Wdyrkf6Syft+ifPOXsnf6T+frRaZzvjZBOe7XfxbHHSVzvHdA1RnQaZiR6K8aboDilZpjpfQo66r4eqtRpZaCbkwGSXh4RlMr5JCoSS8pR46XbNaAHEBOlAYQUgZzR0tgjhZhycmB5V6tAaD0DyqZMylmPKH8wwykrv/eAXJ6h3XxIGCQ8Szm9VsroKpB5b0KqmmHoM5X2W38joXhq6KoAAfdTQFcyqH6oPUUJXJaJsvoJu+8WQph4peL+f01CEyZNsFnFCmX/+LW9Csn4xVNBEdji/I2ht8KnhIPZ+6W1Rys2BUrWovJoSjo8aPbCClUeJRZHkGitq0SajQ3SL5bpPCcJnjBe2IIu4MZxNGfdHYy394SSms0EAmpYmcWIV8SVCFSeanWevdPb9jtYq6LO35wzW2S482fBnZ0s9yAwBNa9BV2LVRVGmUnWIRDhV3S8xCVVnzxBQDC3Mrxx5uJlIOPKpQj8nZ66ioWpEBfmRZAyKblbcEMuGAIGxSbD/izgKIyUotaJ4IIiMBOQBs2zkO7NWHawe2JVFKpki22OwGBVByQ6uXMQicjpIXfWHVy+BK5fz20Vkps1cGmMgj0Y9QIBBSVTygXOPIpHt3HNbvCeIyuwjUQ0g9d2E7VFSOxxdsPstNGmJMpggKlIricL1rc3wHpPBPAJR9gpDVF95ZT9Gh0pgIv4jFpyYOAweGw7EYrU/G2zuxdmADOXOZkGdTBQ40MQ8eYzTJ+3MuRB/NlvxbLXOtsbRNjzb8ienSz3IjK40r8EAUHRRtMqiPyQ9YNX5kuKa6uqpLV8NLNxhw+W9jomvotnBJB+7MydEykBVByKozlAsQTFTKgyOzhz3+nemRLBs7mYYtEWjmyEx2+vrD/ukLRos0LKDc/nRXLuXrx3ArF0AOq09hnZUVCz14bUGg+yBuJiVWXcdmBXUMpUqyJSeHhKoyvjM3AH7nKtsWttRx3M5vS+BynhujA9T3s0/E4bvtbvn66lO9plAVIZ8cg2jn3nlaf0G65EiIk0WSAnnuonERSrqiEL86cKZJH05pIJpxSlY3aO3Axyo1Koj8rtVyBkKCnmJSUeq94nFat+QKC8yJH1AxpLYOeQOZ8AfTSf8cjLp5kAdXZ6WBcOy9amqaVNAtj4go86j5TrbG+I+pGTq2aY/O2HSWaYPr7oNSs6dOSnRIZIEjup9qYVZ9fUEJNTIQgr6aoxmjCsiAsoDErHyo7UhKiAJaJybIUOoq0Gk0dGJXCX88VUP6zPFptyNCX+ofcgdf/gI1VKntHcw+3aU3IuEP1IAbNWU2crMEHbGHWOfhhJpdkxi7pdmpVqafP/+wKFQsoDMs/bPXOIdlCg4KdSi+m5RiyjS2Z1qQ5k3CSERVV1My4WIekxaXZbZ9CKfj7MoYhcJM/7F7mqXsTAjSh25ozHtR5M+pbEuFJpOnnP0Smef72ytzjbG2S4Udzzjo5PTpZ5kykNpXoNipuqhqLlU9YdEKROdL8IB1dFT7VKMKv0XNru+aoZnHp/QIUuaZ/fkh9Oa8WqQxBYbA50rg/zh5SUph9EaAIy3aqSxceJZjg9KPhVlKDR/zGp2POOEjjxoaCDxYolCRqq5uYpTwJvQMzTQTguAUqtSCE59ycvbYuZp6M4L/DX+Jv2aENv72jBp0AcB70dNhuOrJ/0sekrZloYTYMXBPbHMw9tNlcYtSPL9/q6AJI9zk0RAFhoq0ItJq5BRI1OslAaPej4iV85XaHcWeyx8xG4VsbPXNSSUiIo3LmHRQ0ySuQvvyCICPBq/fjCI8+g5Z+909gHPVkvdGih1I25DYsSrex6F9NQDRmM11eNMBHXVvXEJjbMg/UdfyugoIrqUb/NZbkX/sbvA7Ilcwu5gqIOmhtmquw3z3+j/N5JI2T8/+2e6xpLvt1vJBiHeQIM4hIfKrbaG807hdSR5rNGWkaJ88fGeSMVdeQRauy8KkqDstxr8d7sopXkwYSnOlKaSfAddVpW87jrlLJU6HoHb0YFH4GMN4cQlYHzgiqKc2AFCcpJidN215tn3IVw4Im3YKWN6HCbrReJWuFB9RaTu8y1bbPn0HlonRKERrpSIAnxM6xHVsKkbUBxQEzEMaNWLg6HgR885eqWzz6euFYn3ixuDKhDqLqQsgLrnKVEhHjD6jfJpxoFtHD/tiWY4BCGPYiTc5q8otX5315vKrTrsEub7N/KonZWEX9lDaEQGs49CWWyEwlYg9Fo5DnpemreXOYd3mVw0dq9Xhi6gvHrqi4D3eVO+yVFYc14aToYKxB1eB5xdKLVtdSwPbXNiu793yLUWRaZcQbaMePF1HutH4iiEiikOcTpZRDiiXbKlkeXsxQ+ec/RKZ59PXSu8eakbg5ou1G1ILlLd9CRiHhyK29nMTHhaiDjTya7Ih2J3z44itle6kF3/VZ9rncdI3KQQHh1Kil+29HmxW1pLeP4tL16OY03zkipk3XSZjjusIV65XNeAgjjusMY1L27aU8cd1shZlEDJ3IyCQFnic1LT2GbKJ646taiNvh4DO+FQrjQ0DSEquqBk8KmTzlT3YV0m7mG0JAfW9GBO7+dZuc9SFPvuju9q7we3SnQdrczeQV2Wxr4nK250myjc8vQntU3Tqa0RAUkqhtd2XyFoPXCzqk1ihyG+wYPG6Ya8hB6WWdDXU8Dh1ptkyNCO2Rm5CbPemBnUN1b1AvmdaDdDkcJFTWr9Qg1dK3fadlkqu4xIdsjX+RgvoWi8dWheqyC+y4IiadAQYCxFMbJiKW+oPqVa3D8yc7PAxMj4ZYVqT7Xt7xMg7kIIX4dFGrWLUH1U5yIQOUOdUYaXFXFqcMDe7fDst73zRAkVx+2eY9glHevN0iWbBhco3DOrY0NsTPZF32YJYPimjtch6WgkMzGcUhPqaFoxHArBqIhDT4aTV1gJlFkIz+W/G8WZBghtKB1ehWw5ZK/HfBGD3sGVbYekBGd4vVw2fahm/zYHkTjq+wDZksFmKHSnnUkJpiz0TWZSnJbirgdyXqSaOHVEat3DZy/GHZGWyiEn/wJ0JG2GzuS2gSdWFzY5hXFOZ356HtWidD9qq2+thDu5GoGqcDIoXLCVvRNoUWNL5iBoCowMHHXFuKrwctB7uTbTeTZOD5YuXCCXZZKCSQyiYz0YkWYTXuiGBMWpDyhUBaiXxo4Cu2Tx4OKmPGUqwrJOLnUYjIujceHEsWzx2wiwmwm2nCz27lmVvRGyYlTD6MSSQKWIxc0kGoE8PGlGN9SoJaF9Ia1tCNUl4RiMcKUMxsOyfD+sJSdhPgad1dJxJGRCmkFqi8WsR45tYs3TA6MewJNXdgleXDr2Ao1pqzmSsVfBkWBal0G4UIBF488Fj2p/dokcE+B0J8v9kxihAA8q9TSMtFtQfGyqUFsfST6fhfxcHCKcMLsvG+O2sl233kz8w4GGkSJWj+lG2zRGLILNG4IDXgF9xfZum2u5SzOI3S/qw7Y8782rtbZLIQ5ajjKLkm/h/YJm4NhQaYP5OfI4k3KMEYRTj/5xVUUukwJrMBXa6dJPGURbQCBQPAYlytg3blDiIx2DkBSfTvkom+WDn8QXGPliE44DeUvFaivvtpklX22y8ZDOvBR00vdInVGGe+YUVztfNze5wbkxrlaJ9kh1cN8MTvF5xDceg3uMVAG93kAred5cl3EJidkmNmoNf+y6LDL74rqyu2erLZ4rlaaUzFFAt692nD+HnR5NnM/LFbIZ1rANX2jZUoVa0hWE0NjBCbZbKtO0+mFLM67fsAXzZidfJh01FZvkNsYOGBlVTIfTHrGZkLT/Uwcxca4BJiPq+0x23vvZzsprB/z+Obp7nueyAPPohQ3bhDbXoENHpb7py4aQMXR/nQCzU3hODvm+4Fa3UODwFhI8rl7KRulxqk76Md3uckI2qeM1h49rhia3D8RPdBXAiTfQL6HK8PDwK01sUjb9Et0uewULRfdU8Pp8PYO8U5afOWG2oV9jjAct/DvDFB9H7d2Ao2Tnnpzq77/TGxUQYu+MyAZ5pcu4TRdevjhhxGypQPJu6Y52khzPz67CO04TybuOr5KUSXZIgh1qQCHcJ2QpI7grzjnizubp1j+kBgwnFgNdXcIEGsa3NPZm4qtLuWANXo/0Hmvwv/7z/wMtcMyFpakBAA==""",
    "month09": """H4sIABs88GkC/519zc4lN3Lsqwha+zsu/pPeGTZw4Y0N2IA3hjEYzAi+gytLFxqNN4bf3eTXp9WM4ElWHO7UkrLrVJHMDGZGRv7393/++S+//OGH7//mu+//8Z/+9m//+h//5R/+/u+++7u//+fv/s/l/eW/+9f0/V999/3//+Xn//rTH3/4Bf+/8V/++PtfP637/xuv5j7/7//8+adf/2//l63/83/88qc//u6XH/78849/+fVPP//0u//3n/0/pKv/lz/88OOPv/vDz3/56df+b1xu7fnv/tz/+G///f2PP//U/+HDFf8IV+z/7cffj/+x/7F+/p8///SHH3769Zffj7+1//vrEeP//NV33+x8eyQffrPL+RGaf2lYF8OQ2zfD8rhcfmmYMhiG6+EuNxuGVF8a5sqGNaXZMLfXT8wNDd0jh2/fJtfHFZzycYJ/xJBnw5Bff5zg0DA8fIYnZmM5vMevWh6l5Hk5WszKx/H1Eev8bXxLr+0KL6O/CnzU+PoVS2LDluCJrb5+YlnWvwQPH9XH1xsn8DImNMyvP01svIohldmutfB6Fedv41L/iPHbDi/xkcrrNwy0FvnhGpypmozVD2xYYS28Mx7YeNckPItf/nj7Sfuu8WBnnP3seO0vPBc+vz7CZXEa9QLD7F4vxXL08+wz+hKm13umet4z0U8Hse/25pRf2jeNL2CYjV0aIq8hOJvyqOH1t3GFFzFM262/ozO2qY+8iFcFw2T4jNWwpjAbVsOh+szLmCN+VcNpJF7GmP1sl2oUPXi42mzY/OtvUzwvo6tTWOyv8ukKXixjYA9+NTcbpvra0C1bPBZ4x1pefxu3fJzg8Kc6zb+5/o4ZfmrKUpTysHG6YTUOhwtLCE/wxBpf73Fcjdhgi5fUP6pxNvCXpr7hyjfPWHIPKtKhSleHJXU29E45Gt33X3my6380NpzDxUjdNU1bvBu6FpXFyNejTYG4DFhkLEbljzo74/6K+fLKTk0jprvZsBqwKODxT75jxgwfx1h+fmLsL9VmQ2+9I26clJ//52+G0XA4MfFyuAZPzM6IjJGXw1/wxFKd4v9zDxwTSi0jphofZwYb3nc0AxGuptdviPst9r+/pXkRm3WGE2+bMvnwsRb+tYOLtPodhiXYqDEpHyaFDqbhw3x54RdLgRs8pQ7MYLtV48JAzr8f4jCd4R4mo1PgTfc2cf401XIaBMNy/zI5zXbWdcHjG+bwKPC82Or9sXC1O7fZqhluHxyG+zwD8DTrigE7xnVv7Wc/0+2c8ivHlSaW+WZ6xdcbreGJ6Le2q86B9Hp9ciu6/Jh6EIDt4rzk1mJ3gBV8RTDwfqQndhhbnOJkcmAPnHzEna1c2roDjvMnrc9g9eLbBHbAtLGLdNnv/jfN3rA+cdyLI1jZ/+bphtENczDus4v/rR6eWIwLLaGhfpauCX+NbRqKcmvLw81MP7XvN8M/eXQXfdnm2NSdx2VAhWyf3m5l3IR8odOrPSx6OrxXmM1Cifdbe2SV5ovlsDNuQbAKrgzIGGe7LzHxhd28QV2/LZU5RrT+zcp9cHljm+Ea6Pvao0/rdiXMPi0Y5wge5/ploMFNrQXh/jPsfHPSDQ+Wz2UISLn/0QAxdbn9zrusbx5j+RhUdAfgAFK2kBRkGEf6Epy2t3Aa2oVHmxeigxEjhVECR4niwLAYr0iJgQ64Em6163q9FsUx4Iro0bwBnCjT1gFXgAfGIi0iA66RhAxSlEjPG+RvhsYtrVbGWwWdk+VBa2Jnf7kC7sJnBeHlkdpT3Foiu9iXFQyT4UYD/dJxn8uz4RcsdXtlHgmkBk8sxrahq2/3LR7fsRTpzlwi5Nq6obH6dBQH7mmwGFWJg0s8q8YuTYnimfi0igFNfbvqKKCJXxMws756sLPf2C4F46C8P+Fypp8HhPb6wQXMpDsKDNayX4IdrftBjNW6w/bwUXx6Xt9+i/HJQOfgXJzrX2GKgX3ruBTun/eZVZsxWg+6FiLc5/+SWVCLS/4Pzl7qm9xAW5wbzRmSapdvUoqzX3DmT9rhgbLy3bDvZEzjeCNxTKA+ur7aYJiKcf9gPJKblnOgB0ZIcffd5gS/9Hn5DAAOglEwrHz19NEpwLcuiVFIU9WHkd6uS+4vJED21q288c0zBQzxxn2uLVfPeuGNx9gzLXEWx0X0vQbCIxjDaZxRs2pKZbN77SuBYYtR2Gw5Q6ZjBNKsVO9G3cXl2c4ZGaBE0MDNdcY66pBeqYePEig8z9d8f/n8hBRzMa2O5LdQn0RIMawMN0PQQH5aCwQptLfDPJX8MVsmSCEtXndgYKbuFThG72xNBDDyWcAQL5+9WOk+Lh72gs+TvQtuMNmbeU7WxznhN+po9fagj4xmmd/O2SjS0fU/lQoFPivl7giJNA9lOmdUzSlt0KPPBQjGiJkERPyTkfENh1zu/nELK2DgEAMxrZVPiO7p0Vq732ZLdb8DkRCa4skG9JlBWj+NJSiV7zRfjrL9OykPkzuawJS0K0pKOtQnevyWkjaqO1Rs6UgrB6l8RbnsjrQC1UyqEjRjeMw/1K59UJSO/YteUN0pWSgrfCKtVsA3WaXd5raF1u5ksgFhPCdwXIUsajSYHW2fwbGvnfRT+8WszEnw7u6NmnCru3rpCC/VMCycwpk84oiCUXrFHMDXjGBtIHsCorkf9gxRPlnOlAy7l4I3zJZdZNAUydC4D1I6reMKX8GwRK/80lG6ChDrSyn3T1xgUzGCE8Ef8WkESNS3YzP5azZIhYurx6hJ3CxsJm5ONpPPgqMihnr2HGVhxMNe6y4PY3uXRGkY0ZtBCvQN9wmUmjf8NeZh9ACRCKT5mTE0SjtKWb2DNEDY/Sul+2TKJ9aauKJlxMegcAZ8wUzm4FJJHNPBSoCkj8FPWui+CUhGvkWlzjJI2w74KSa/OCzMvVCVXFEpDNFScApmYq4gYLTypDje1nU6RksRqGLNWMGcGKO5qhTjs2eINlcRu2GyDAtDtAK5IivVzjmtDnodnEBnUFopZPYjmL1SrCbAFEtfm6hcH9vChWsB7o+13Qejz6zWnEYbl3EjT9+WtFZOQq5heWAPtjkr6IVwVhoViwQBsEThgXlcWeGHVq98mR45PWZFmrQSg748ZcFr/3uT9ILdcV6+zIbOuJo1QkvhSX37ZmhcsihR+Ola4In+ZdGaojyirG4UFbR09iz13VaQJX1Lxi/i2q0gS9kqbHW0MeVzwPjxjYNXKcMkHXRORuqeJQSCEy1g9SkJzSfDzjVg2zkLT3CuKM4FD2cyrDPalf7CcbYLRlUuYJUs9EuL1LTA/RUd7c4pu/SoFoc4bpprPpngVUjOj+6KCHapSNWA0Vk142SbC7PtddnUuionw2bXskn51CWnFZNXAEzdAaZnK9F9Z81okAHyYrbKK57xUsgSAZVra9eThP/t5uGkVEHsp8dLZXiqdsXYj51XaAYUi7AMOEJMUOBEHBjwvnSxoJCRjgEnmlxR7PDmOO78LzsHFrvc3wdSDM1Lz+t3t6lvZMSxl+uwgBesXoxom5TPmWOPSXm2C016XoG6QLdLlwLORkZ//i7dToKtJcwJ9272GiYvZt0TejAT3m7BO0WKm+KzCAaqb7Y87ehDHq7b4TY53JWHh+DwzMlHnIgFskthxrAbraAzbCklK91Pg6WDjNNo1aQdwwEoreRnc/nt9duPvvOoUFXL0m45g/gNF6UuOZSQpZoMxa8OCaqX6P4LIgg5SCG6bBGBRb2vSwqlUGrQOu6OIUHC3GDK8T7X/QkJkq8K7ZANOxBvsL9dzFJqIj7K7AnHhixKiiGWx9W0m07hZEgKcGVpLmnJEDfVOMVb42cupDRwUCFGLReSIUQkLwXpC+gam8iyNIhk8PU1KF8l9z2a4O1akkpqo1m5Te83+p+kFEqPgRNvsdt9uZQL0X1uQB92yb8b3se1MSlxWnzYi/j+/rvJn3LlzEhLt1Z/3t8o6rbkcHtyBuQTx19fPeGczFA9Cl32e+gLD4y1xfidRIn1EbMLdi8f8TR8hszXpi+EaAxDheGKEvFzS36x+Q817eRwNtn8GnaqNrvr8E6bZhNs+YEJCf6WdgPdhXtwr7FKXIQluFcHu81Y+8ahvV1ahm/HRNnVzRNXOWLTUt1x17s9ju9VJJbG9ayifXMX0R/c2/slLhXt3h7wfmTGvrjrpK5f8ym3a599f8M8G3oN9HCl3z+J47fhvc294t0sGlR4rjr4fgFKs2F6mTd9EaVnYZpul5UIyFE6+yQVHaRnLUH66N3UT3lD0dAWTt4oa/lA2pjrZVo6CDeXaevcLUUH8ZxTt5LuV1jXpwOqSxOTCBzf54rRrnV+CfAek9ZFvIUnyPTYfRZlVwSw2YZrdMceIqlLbaW2blLPcSdctbnZ1oX+4JI/CbcJWWMWjX0N7w0LcRaaKMxiuCqE6WqdCb8hmg7smiRYMDQ2AJhHjYuAGXkNzy9k0eEptEu0eyZ37i9imW/fXwgy3/ynUYi7oXyOSmNQnsh5TP8oTrsR952W4YlVShQQ9893tFGVG3+PY3FyazX0u4GUJeoOcO6sG4a5vRuku1EVr9LSw5act/h2a5Ff+ZhrlD5YOnmrrClvaWuu12LpKLCZfvTKhthoHnXmX4qeZb3bXsCz2Cg38S3cPTuobnVqqF2C+13sFtq8vffbXY2k99dhwZzo2XA3yxLfQ40SVW25989Yd1fqTTu9v10De9m17drXWw6444ArAfcmvmuaDuFryuWWVcU/E1t2zZraShp0/d9Acqo0qemBWm9HdtCKgJnv4R69k9VouN7D5yZa84K0svjoQtD9ocJ+X7phh9vWiAV5EGkguph1i7Jrau2GvhatFt4wLGkXcWqn7B+qSGb9+jgni8KzvnMXPTlWJyWhrD5speO9/2qHH/Jw3eR9sl6MpX15czG2zgGbnZ27dw462smepWDrguzKiNw/2HgxCBqL1IHpGjASbEG5tECKVJqiTUQybaOlNUnCW8yO+wpTb7s6ygINQlU45VQL9+kpxHyv/LI0WgC+3lR8F5HsJjWocaC+0LtsbsZLy0QtTWJA5w00sIpPS0wJBWTJRoXMUnZMjA1K1qqGnqvvMYCbsepdnlP04dKuqpGJdQErjhY3si2iXXmGP8FU8+aUeepwXnD3K6ZA5eERlERoMGZFQFwqXkIxY+YDRqYaqxbifYLw2aQS/GgMn4NMeBJe37qOx+dvvr2Oiw9jXYezlzv8lu+sXaObtbpZWPHpZHe+cxww8ornjz6Let7ZTHUvcNd5w53BKJQ3/KdDf026jhspZVK1d+WZQbkVjk0LIMlO0d6i5jjXUMjMlhliPNKoRm2kYujW6Uf9RxIeaWGTqTCJ14u77rBibhfd0bp27Qh6tRl5Brv0d9m1FWzu420Zn1Kudr9L2VsjqOhnoqigIjalEFvbjuQ/jrzlmfyu7L9J1+7qAsGU6+dkA+quD89rdNIyoqC7Uo8QTsrTowy6GgDX/H40xXhvavDR1vpqWx58NPKFL4DBLE1eh8p1fh8YxKwoEKgPW+zEl1sUD/SvWXY5983yVaKiHWwXeXtyyNWPQ9kkz63jx0/Tzjpb6a6F47voysBK9JskBucCtAfoIzJcfI42umW8JTzmLj1Q/NNil1OOntHLRpe4MnrJqCpmsNLpQk2KnDv9ncww5EL589qkCqd/VpfvpeQWAka7NK73wqNIGKRNDtoycgyuK3ZZfIEhLpX71NaCCi70niJhmCXJd4cpb2FItTQEA8OQ4qV8OZcvUAJ9ZHkNUF/rTsp8c5VjQxS43d1Uw0ZYfAQWL0FJkv0dEVBq4yOZ726XLyfWIOZbRB1/dCL3ftZt7oY1KuwNUpfudu1K7wOK5rMkoHTyrLNXO/2U8tpxpv5oq6g7k+O0fhAKoQLp5K2wQDrpZKZ6FjZT/Ri8muw14d7+hpemEUjOQbfhbvzZYlgvaYYOdYU7/0C9ZWvIU2W4FFFEx5r8wWAi8nCoKiVLO1qKOUgCyHULlzYigI3hErS9bERtCidtoHRo93CSoR969FnpD2l1RzzdMeHbLm0jthcv9FGzoseBgSatbgoQbccD1bMTpFS6wyF1g3usWvOrposLr0gm+yLuaJnxWfy6b6hEmbZxLZZqSMy6GrcYJ1Eo6KJqBc5XBRZgo42RSl4iSpKMYHrOJFVElDI80Dep15Q1Gcckg/gufOlGtSh5DfFhN0qOm5drO03GzcfMOy7hZvV2KokqzlK351q4EE9DJqKBePgw1aAf9mBjEZHHonsyfDfddSL0kV01dADKkWEZGXuNdERQBrhmtis428KaNMpI5OLpK1J/nHPQw7KhbDQGWjAO3R4Swzgk4AzWzQSHylDLe6FzkIN0h1rAYVJF+QhpbTQvVqQFVVi1Y74DrS+jWt7s46Py2IZx3JbyWAkCAWp5YMTL5oY/seu3Ubnwi3jWJovSwk4Fa+fSNgrtI1kuZQhJk2pXDHA7aandPTzv6k5jSpA7kHpK5ujllXEK+jH9GiP1jHL5YUiBSqeexXjSc3zsffUINYOSJiTIoEeiXYiPWrDL2avJ3/JG+sdaugUqne2Vs635zkkoBF7Uo5dt8GIfdKR4iG6F3u0NNxZpvqjoNsFK9tKVx5lIUWGZrnUV0JHYqCzQDPGrIC1ko2hEIKvy7DhrxDa5946yHDJcrXnCFFC64dxMtJlNyaBg8DY1EeNtKszUgV/QhH/U8LbI/Sc+Ay6YpRD2KqXlkVIStGwIAy1Vh8BVmO1sdmi9wFmQ9Fab9GkEucg+W7JSMjeEVEt2bL4tKUi8/y0aIjsfukdLRZsSwpIe/W+RqoZUt9+kJupWYaMHTomhypf+HuCLsvAsK9DXM1VNZRsAfe43JokRSwoN+SGxPxHz9Kh9KWLS2rNuZB2sN1swiPgl79I11sotGZSjnXK4MeVzsMAQ6dhx4kU/5PAjRafCkOfMh+k+s0Ya1yG5aBSCkCPCIht5RZDAGgUugzhNBa4rPbm2twUuarG5EoiAjApXahJW6ic0K1oX5DS73dwHvZFxJO/eUV0Wpg+zE2NMZ80YXOwaaMhZY3YWMHAhhV2doODc8xr0pghxh2UkSSulXDoqy/5t0bpPUAaFeq35fcVkWiveBw8LNJn9L3JYMC1AJP4tzehqY8bSVL7JrZdda7iequEObxXtoJfvt/YgCVtyx7WVXVjAFXU3jqyBlhqiNkwJEaw9zJZk7wtV7gsfl6P0O6nvdojNlgOUJE1Dkx9Vds26m1cru7ZbW/14yfAoK7ckeMSdwgBE3JgLSlLPQSa4I526pc50dsxVt8Iw6cyLveE2a9jMdN00YCWcJyKGhWXO2OWBjLUZY0GjS7vhjMRHPcwaLEG4JeAIk9qhWVSoCx3TwWS60eskXbuvr1Xdd0tiV8Lp65bayAJ5EmRmzTHOi13G9jttkOEnoks+vz81iiCdPHjhao8Y3xfbokqoqOa3Zto0faGloimqHXwWNGFo0aZ7cp8w2/T+7Pr2dhTftq0wavSFJfOlVqco321do1+QyfPl3g4sjLCy1AaytpjlvnxS9grrIvbM6BumUpaYwWvprfT3cyf9V0XC8gSUxlxKARSoz+IA+MbLbRND1sfkvIS+eGXXM2TtFa78aDtzQUpnB0E9d0t5SjvlS4pHdCoMO3UnFnfQxXaaBX6m7qWpu/q6cFxUNgsprrFhbJJ8anJsWCvqvRjKhjSL5HLzrt70bjFw+ToX7lsZTbPzoLwzdNmy4uG7Xb2wiOaicve+AgoHjE4u6YciFpSLaFfEPkZp+OEnoEs+vj9p6srQIKEOsbgGKS2+q5a84jlNT+ETz+UU3+11WNoITHmKF7m27KXe1boj9ZttKrfATG0wYoq9RbFYgBIR5a0yxZr8IlqjdnVfiesq5uE4VoKUikIea5Hme6xc8CJVtFZGd7/tuHRAzB59qeVtsbmijQbRn1V2fOfNu23VTK1vueSGtKXb53jU1NzZrpQPwZJ00U9d2cyE3xGpyg65bFoUGLmobgyfp7rNgjkl0Ut3oAR5k1afv+tO3Q5rb62CgvcgDRnK5lhBa88+8W+UoaQocXWziJ1l9VIKFG0owArDzJZYeYFm0a7Pq7JhqQA9XJIgi6NZo1kbQd7tahV6zV9gueQFRdjFLuAIZbUSxlBOHQXS7RqqQGny5JSck5VSiUUntp+uoCyqmBoZbXobNqfLrBaaO4KZSiFmN6iycLi4L9al6FZr3fQXM6RJqJgFA1/RxGAWJlQRM6Sc/ei+M2RRqx/u7LX70rc194ZWhNKPpD5sUbM7e7uzjymuHEOPk12ibskXuOP9A3B43OTTvYCVI2/yjveCtVa9Jc+8U70zk6NbfiBPKBkw2pFVqtIsdlTwbRmUhgfZx5gzg3K6rWDKytTnY6tZJ3EwfYSRE8OsUCd7UQZ4tEqD2bzSuTnMEgEcJZk67EAOwp6OSD8TAaM8xbvjtwspbFoEua4HilIrc7YW2KcN/vhEbzCGVtVzp3KuLBLL6M2SPXuBwoDaIGvFcI1U7tJmz7LpdGrbsqXKFubyo0qgYQe/KYHsq4FqfoXKC0VVZ8AIOyQ9/EFrXL/oBSnnyyWl7jcUYtECcrJL74/CsR7GRGPt3bZQxfqQK4NXXLW2q+5om+R0U8qHYIEdR4dOPuRsJjoVNtOdGHwU1WmSSprspPu3wxxJBGbXZjAQqhV2u1Sior6DHJ9uNystb2beNn7e1J9tMpOXiJ6giDuYyUEzw1KQV9xztwI5qVGzk2BO38GXpvrfbIA6uDYpaY8r7n5y9/I0BJryqJxul9rb080YMm7kb9omxagq0zPyU3VrKeUnV48IwlnSKXcJOJUgzF5F7U7ihJhM2GVnKxNY+E5pFT3ukJGcHqEL+gZy5C3CyS2dzI0aKv713WxMe1wKlVl91r6iY73aPtGx+Y5bdupm3ba5B22fyPtyjebqOSi7aL7pnWsUzQ/O+ZlXkX0Y/UjVZWLmQfXPrFXcPEyxG9mY8nqfxEh2/sJ5CQYHPVe2azgvwXg7VKtto1cImMhNUntr/oHSPFGhE3erfIX70YWrWfGCvA7TpbtdxT7zIvihjxZQpMo9vBST+/2gvk0l7lYgKmEJX65mqWRpwnoju4qD5MyZr5nw7IWzYqVpd4zXd10HbYO7RYF9xs+itO2ChDX5Oga0oijLgky1ahEBTJG4sCDFInbLkW+2SCB3yE2r3qwAbJMR2YqV7kDDLuXTL2jx/Rat9mR3vNdFrT5LfbUbzCB+SnnpmPUq7hRaAHVfkiM6OwXiiWMr+XjPUUf1JfxioueiVVP9JD9NdstAwNGjgMMoMJ+agYWMsgg2ZqGVrbO8NbJa0GvdWbUkCBSiUd/PTsrVkFUtChUCrAYAUsaILlbmxCf7Ywz0o3h+NtKoRGi1mX++tcpK71WjubtXff9R1rTevdWGFJx3VlEZBr9YNZ/fthKVdxcrK2e7s9LEXxaj8v7v0/qCFiMNtkt0kxuj7NLbL2XlUvabSQUqAG+ef8VdlJQehF+iHaSH5I9XN0YmK8h+o001qm6MlK13ssdPDlM7UJzUHMTORptGojm9vY3pXk0b25HjrKpaYMDxLo1DZmUu/2yyOJntkDlsZnHILLuoJHEamSWnDLbG0DHqrFjbqllRJxmHosS3RZLHzSDUt5M4o6xby30Sh71tt6v+7SzOyLAGbb50Jbvo430aZ32cQxz+OovDZgnnfG+SOJnsTiYTDzt1cCF+FXFA0bIz9cEA+LwTHd+PeiJP91FlvRb8iUeN0h9Fa+VhqyMK7kc5Ipp85LOyz0c8SaqcoCLtQfhW+ahuVnRe0LzxyxmLuRz1lFW9jX5+WD3SKTo6Y0fHuR5NMKlHo9Tq0fTYN7xwQTPR6WdYLzHEjBlP5POBMDPmf7/ewtFvYsyG2IO9VoP9TaKDWZCg5thkEnvWEIoFLJPXs4toFq+HHxbneacqrYd3skXrYbO+n3A2vFPotnTYVFLP6FVq75N62B9sSD1sFy+B1LMgCkey6CarJ5OZxy50idVT9YHN8DGLOJmRv0o5Goj0UfShAnUHK0R2wUfRterqDo+opJ4sNjLf4BiLGkJm6YRnS+BH47x8+KOC0kGuxutFuWoCLZGjlGQOdt3gsw2PZ94bWe9xzxtct9mJYHa28d84Z2AmHmuCrEcT4w49Vj2baK+6Y0ZOqvMHNR891DhPYfuaqb99sY1mc9Q9HNTpCoDLGsTtsx24N+1iWJQbn8FVpV8ss908vnvTL4YcHor4Vr/YejdPOEc9aPHeP9NEbzaM0VaxGsbWXEDM7f1+sdEA74vUL7ZNIlj9YksAbig46m2pjUJ2ONjOmgWM6IlcitUvtjyt4Ch7VetwnJhLakUgu8TiYlK/WNHnYePjDmd6Fn0sFnzOfDZZIusCwW2HnrLGf/hIZ0o86airPR50N30EjWnMjCiRxbPBamJ/2NGDgkyfLhuAp3y/o5U62xWHezCfaWJnfeBI2WBJ8Twfug/VWxEoPHKNbzhiwq2i3wdJQjnKuEwIwcUyw7Rk2AVnI4Sh1Gj8yMTAIpUgKBeVtklAbJSLGgEEJJEHRZ2YYIWpW7SauRDvZYteoJH57mDKFnHc7X+/y2+LFtE2UTWLhha4c29LFvEdTFQs6tuppPdHd/AxFfWKSnrmgm/kihZcR9dZS65osfM47FYdUsaeUpQrKhfOM7H0Phj4iOO+V7x0Mu00n80Ly0cDNJKmSMxWR9KK8UDr5iNoDeQ36TAxYeeOKO4HfLGzB5291dEXPFmro11xtAGzPHJklxjcjHNLiAEPjvGZz1Ad1A1w1Nyh6nzJSnb1MDREjyyu7RDImAj3Gl15R3begV000j2FExuQlLJ1t1Hh8XPsRlNktxfI41xUVLcXFJLbvej2Cnm+pK/2ktsrdCnhfcHtMc+u5Xu97RXxwBQ+Q237BvCIWtsl4QgcQ2p7ybzEvp3yvdT2gncCqM+LUtv9oOaY31baZq8gCm13HxSwL9YpfScDJYF2m63git8ylwcm/i+lry8nlF6x5GlvwNWmOlEJXaljzOGYprOhseloONZHOhsqEk90kz/CkVZlOJN/8qdiDAetjifC2ifPOXunsw94tFhH++JsD55t+LPTpZ7lfQZR01c5c1NHLlH1vwviFL19bRvEacUW5oIRbBliFkUzC1POpWQz3ha/KRx1uxra27mazby5pU41pznNcXMr4klXvB82t4c85qi51SyUdj9obl/cMsfMvYBKVxLGzC12lNW2xsyxPFcZwxDj/Zi5u2KaNWZuAfAXMPmc1lO01OCsMXOLXeVxcSlJNb/M4+mkMXPdf8GAY1Gpk52lNS6JzRxMotkMZyq7QqE4YI5SD+J8uaRNaWeIdTZPN57MlgtnY1bCkWS5FyUXb2qgmvrRx5HixUn73slzzt7p7AMeLdbZxjjZg/FsSq94tvY0Qe0cn/kM3UHtEo+aNzxzvaqfH+XATbrSDisxEDC7pi63HuyT8UVQNmJAqimB0jFJvIwabSCoBANR0qMa93BfCPRE5MQ547PUQOilBdBQz5I+VbcLCQqnlzK0toMekjJtVUsrXQXSek0aR9+xEub1tJYHgkojZarc4TtSKgDofBSREoxQHPRDBYznoe+GWqaXQp3vly9SXlcwfD93oQiDZZZfGXEEgX/+NW8CLGuU9gqwMorDOElFOzWcSGhPCsdvmQqqB1qNTYyxEqrYS2PQP5JHKSCN0PIRK+qmWTPet9DMGijPRuJsVYZmZwP3/NHsGn+oCC8KUzJVRBJ92gA6Q1uBGuwOnnL2Qmdf72ilznZF1Mf72vw+a7tz14d4tBjQHR1k0Wnws0QPRQFedYgLfJTc7z7zaPl6hnRiZCGBizJ21QywwiUBOqzvlmRyn33ZMIBG5s0grtdGdjMFYCT6JNVKxi71EbISp8vIIwGBrkiUcCzoD6aaoJwyMI+LWDpNJ9mh/vTqtRqcy+9PKmZqxGUUy9msH+gGyCwoQ/H4HLhHSQrK7afuy9640SK7IVRZDamMQioOi/ZmFw3eGOgW2297SctGwXwhcd4fUXW+9ifdwblYUdFWlG7F2NEXvwpl6OiwS12EZUGbQs9W4lDWm+SXSNXzh7N6zqTzTyRpD5SoTh7jdFHy+b6rfj/avmerdbY1xG1ITkDd81tsZp0vYiypZ5m+ve45Iib2JDe1z+xZPpF/ou6By4ZUaAsQ1IiYToouS+ZsZAHdDJhclYAW1sW6XXavt5aPBJhmIvkg0hkJisR2X37YNyKd0QKD7Q0MtHJHp+1e5LUDpnYBHjTXfFNOGzw3JxQLxzx0DzS6rMx8GHipoMCaVEOl5OzQZVPEQnIH03N6rxktHyvMyul9VTaGWUMoWHk3j7Mtrc7VFWXliOk2pfOO0/f9ECpP67c9j7SFoCT3UsK5FDbbgToYA00Okup36UKZ6b4eUq2w4myD7mMlzQ+OHk5JyvZIlcP7Iv6hoMiLmKKjapdYqfUNmcoaz2fkv44mTB8NdftwZ3OUNP3yDajTpEJPHqO+0SJQezT58my11K1BDxP3IVuJm5616cQjRuhHPc4JsaDqOgpS/SRHta/yinrdb/hgyu9JHp9ExNT40lFdw5qtFKn7h+MCnpuiYOnLZmB4z9jMAxa8jI5VxIJEFfwk6kfhZyIv0dTt31O1RxNpldBLCpg5a+9LNaktpB29+AjFSaf0KjCVtINgVyT0kgIgs6awmZnu2q9iV3mb57RhqOUdB73fg5XKZPIwz3tIgyg1ZaYN+2eK8A70FJwftWlWKDbosfR/1pRUQ9kgDWER+dq6MzPoiSjNohVBmR4ucuE4NyJOj2QKu9gZ644m2TKbXxxVeTQA6kAi/eg5R6909vnO1krdGHSe1W1ITkfc83SaxfPFxUn9NAcTX21kIyFDJ7op2lGiT+T8l+yB06YDw3L3pLUhB5e4a+AfJHFD/cKx3SzSGp9c7zsFttKfn8HOwmXeE1BKyc9JLGNEdOb0UA2S4Ftk6tGVQYHtMu5AJRHCCgXYbTVJrXfxQUkzpR2UIVY/fVIWa5DNQBTkUgRX88iJ4zxKC74EwlgtQK6tNi+V71J5W5i3I7OrCRy1Na8U2/tSIing5FL3iJpowxfX/a18quWHGs4hE8dYxjKPETUV4ralQv+oXunBiB7nT2judRDlQU5ZTLSFjBp9Ys01eBSAEblwvkLLsEU5ZquI7SVie6xryBkRBVJcwtKJmNZzZyPLjwaBHsyvOXrO2TudfcCz1TrbGuI2ZDx3tOfPDph6mjnNJvqOraCx5aiIeqe5RH6S6H+J/qW7++wxOafEFpbb4PpbB+5SHwBXxPrKX8a0yrphHpWvw3ZuxilRT4XZD8rVxcHmB3EPKV28IKXS7ylOKS/65979lmhTqEfYnTK6B4r0K7HwbenwLtxuTJGOBJ1R+K6esFLx8Z7bts9idX8uFfy+siF+M4tSgg5T1NZ4zT2rqgPXpIjZc7OUpQzHCLe7E4/osSoOb8FYTsp9+XmimArnuONMzOoRe0asm1J1RmTQhQsVYDQm75IwEztcvYfmElHshGtjotwd06tEGWNqfoyaUNvRVPeDCYFHzzl6pbPPd7ZWR/vibA+ebXjxcHEaUDvInM8TvcaK5N53UZoz5CfJnrdummJtP5+Czbizowpmyhh75CfP5j5Lkzykkq4opZL626QiqNky9Ege1c+UccodekQk4/uoQYhWALFY6TUcRJXSfDg30zSxdzSNo+/fnziQ+hJgMsm8kSTCA3M12BzzRJi43+wCDh+T5g/2ayS0ZYrKEN5hy4CmUvPhCrLLxJyL81gxDRLr5aPiHUic8RTxeGt3XWneLUfAg+ccvdLZ5ztbK3VjkOM524Znez44VCYUicUh4kQcTYcvFJhzItbR44XzB7VkbY+c83Di3U3Cm7HTapTu3jRg7BS9VOA7cWpeEQKtddPyPDIFQaDHj9AEFN+RKnD334RbpUeuwBAQRV33VB8ZNacspYGaNi3Wm+mKFGWwXXI0tOXXvzM5uhpnB0HUGZNWU9l0Z+/0o8quHVkULnINyc9iSYCbfi112zXOXO39YFiJCKexUDqIzvdDPzdhxhqRzmHm4Dlnq8UXGU3NgJumNxMjC+akS5AYgZjKpm5ruz6ON6AKw7HUqht3adt3hUJXoHmo/ca5JtuRX6/5eR2Glg33bUixuHjfLzPsLlRmNjLF1e/SjXZNPe0UWD/Hc7x+Xqp0WahVEnRGAek0eEAw1qMYrbvkx5F/UarF3MWv2W+HBW5C1cgwI9OAqCW/TdB48biyy0SJmsIuQjHeVP/eE77kcWF4f7LG9HEiELUKrKnI5IMSCrbbTfbYwVhBsGfTNIaui1hVNisZSkGUu7E5OtBiHzKOVjQrQegmKw0NtrIVmdwdTLIeh8PQBeVEQIigOpAM34XNaVhMGDNWDHAd+Ww75Nk4w3chH4jaHzciB5mpF7UBtktF8XmuANXGFAxnTMgZT3FqjLseyByTagPUMKy117/xZruO4Y3aZkI0M7cH7kSd4gbO2EIFhe6JCW9h1rjP1Owcq1Wror7O0GAmqan40K9hmbZywqNTjfCWOKkVLjg63rjepExHIAdJ5yMU6oZz9Z7axt2urs2MuJ2rRLccUT2jGaODwS10IArTzWxhMzjdHYjOky3Mkib5ylBweK2dVYR7d98nETtaLIDhsl1B/QR5VUm1jiMGdslIyOMMWsqtbxSbMUtOnU/j1m3c1p2jNKZPThGXy1v8ZKsMg2vgfjWrz4T25TNk33AZiU1KaMEqtI9th7uyzoQMO9ntyS2UnDCAC/ThDrtAtdxqbR5uf/6MBYepWV0+jGku1DayqKSUegoeNS5tER9yXUOXFQYKZmNOEfouuMVtxhDCAkTy5z2+SnICBEU37I/gN3Vzsz+LP4ofjLOqdNYhg5u7HmxFJNjO+s8EgO4rijpsyMrexvWbj4my+LziwQrFlQ4PkOGtzjoO/ZTbMdUL6NANuZuaonIOAG37AlNvNnzxZCeFbDITBEfiRGotirxHLB78ukeyU1hasGp90bCH0sIzjb5hmsOpRSSjEOAbjDeyy7Jud2Zstlt2+D1iVOagwpB7n55yjLeUvEofZJ4OsClvR45TYWoEye3pVe7CG9Tuu1UyWIo+0Lmuk+ZKt7uMXudAtKk8a6dcZooy2LypcpnUXvTI3SxE+JFWocNF4sHPYXEwR3MTlAKje7rup11+bpsXvxNfD6t22fQjtAbYHjaamH0WdH3icJtp/pnOULYOGwWyPNxCEoYPU+gYoseWkHa1L8Tl6w3m7hoX+8+cHzdI8eFeVLLfIvLU2DTa5I01T4HeLky+ob9dMA4QFql8fcbr3+yccZWGveJcf/60COOS6u7vSJimzEORNlsH4d//538BVhvAtwFwAQA=""",
    "month10": """H4sIABs88GkC/519Tc8mN5LcXxF03n5c/CZ9W+wChi9rwAZ8MYzBQCPYA2slQ6PxZbH/3eTbT/dbEaxkRfHYUmfXU0UyM5gZGflvP/7tt7///tPPP/7HH378l//yj//4H/7lv/3nf/6nH/7pn//rD//p8P7wP/z39OM//PDj//39t//317/8/Dv+vfF//vLnPz6s+9+N7nAff/tff/v1j//d/6M7+h/+1+9//cuffv/5b7/98vc//vrbr3/6P//a/08a/+enn3/55U8//fb3X//o/8Ufub7/29/6H//Hv/34y2+/jn+jlJc/Sv9fv/x5/L3iXt6NP/70268//fzrH7//efyr/X8crxz+/R9++LRLr+rS2c61S7PiT2ZfXIqvw+dPu+Pl8rVhQsPsXikrDwxo5l+1wfsFX6TnxZeP7WwYU7w0rPSG4RVDOr9hKv7S0Ef+NPnjEd8Ny9dFm38qvmPqa9Hq2bClfL2GFQ1L//oZvunHL783rP1rwMcJx7Vh9LwcscSzYUnXy+EdGoZXTfBTazX2KX7VnF7+bOhfx3FtWPEdc305586G3l2/Y8u8c2qBJ8YYrrcqfZzyqmezbKxiKGjWXr6Ws2GJ14YRv+nYYbWdDau/3uIRv2n/m+GIYGicYX5eCqc3DC/rKCZyNT7ks1kM1+fCo114ZTAL7frtyvmzjFWIxZ3tjmbstAx2qW8E+CjNeF6dPov/8BDfH1icsX6B3i/C16zx+nmhgVk/OgWeVqWnyb8yF/qczqezXQrXm8XT10yl4aJ74Wf2w3ckeFxw14evol3fLLDoJVbl8PnWA0f4bphz/9rXv7M2MAz5dXq/foaDd9e7E81Kj6j5bJetM4u7LNRXbu5s2Pz1h/HoB8PYsJ9ftK9nbtcLX9G9BNcxw+eB6P70CMYr0jt2j3k68N0wZC89sXuUDE/MhmOq9MTYfT2sxuGun1hoFdPrgEW8/qQlsFU74HHZOLqZ90w9HfmxhsmILOc1dKnvSnRprd6fiY/N3VI9L32r1yHQZ94zJXhYQWOXEgbqeyahYc73Pu1jy4RUznatBXXLOOUYtsQ75sDjZC1hC8sd8zVmXGyZOm2Z0M4Yz5eq4Nj+RaOP5yf65sRP6ktUPA3tmv5JXfLKLr34pmcnbL8jRVD4pgNlWNsbrGJHrRCv4+GFG0UYYLcpjjRl/qCuAr73BvZtnj/oea+NFzSwb4vsL44D7kwpGpcmOhUNIVf/NtndO5r+Tcd/AUhZkhG2My9GqgAvjpBFVxMcflUDl4TE6+gq3ESTgbfpgtfX8ahwFKtxojjkd7/bpOWIq/39xm+314mx+tGf18JZSI8hhvNw8YlGwKc17BDD461AyQj0vRZaU/A2edI4DjoYhlqVgxhL30SndYj9J8X7C0U3rD2s5bOh5b0j3ZfHP3V6Ynq5loQbU+prfV6K1K9wxj27LJMQyXTedCpS32oNDKsREMlldIxRj9NPzeY92xc+wC7BOaxRWsZ+gAHq2ymoCZe2CimhZNxDKXmBx9C/0lGVa/Y4h9Up1+w8YX1/hPMGdyFKAWMgZunG3Ca/n87pmX4NMhJCZfL71cPJOLy0UaPH62HswdXwbnSIA17TY48fxkZF9xbjq2b4qTUWKbZ1FA1n0R3Xq5Eiu5t6zpakfuauf2nGeBG7ZzynIPuVuEp4KA0kgIfYReUelPqmbh4OsW8K5kvx5dvZzocg2fUnnFNC+Q1Wb79N6h/1jN7yKxvojV1xx7GlnA2tLGsMywR07oenKUA6h3cAfRuW7rYMaNM4dRkzGDoDDxMm4hRW9yM1Sk/s5/+EbbthKEkJ/cOJlnA2jOk+ufDk0+CFVl795PhGe7SoZLzjZEhBI1avJOfDcLLwxGp4xhmf1gL54P65pIDaA0xpCn4jlzrwqZPywXmRBRlhynCobUpnOA/xLV6+YY/zU1xsmKwrxgWzcFxMZ0/co1QpUn6odS8BhqFEBWv0a81xzprGV87SZTGOHBiEtxa1aDPywP7s+637UEbQENO7iPOJNUtTslk9LhJILYbzZwBf38nS74btcjl4/Xt4OxwEjUuIOpm5vm8hSF2nlCc7wtLd2cTro98qY2nvwbAaqW+6t3f39rVe++kWjRQDx+/az5iHiGHVIKi81n/DuQxcTIxS8rK+2h2/kf8uiatkLcBPzUY4JWfTgxskNUZCvAmZ8x7b3NkRF9O70f4eNfEEhs0INlR6HA7NwTI24/CHQCUT8Q2xsqN/0oSVK3kNM0ZvedNAXU7fpFADfnAqoJKkn8KIj5OPPd2FXAag2FclH1kJFr6+asXiuAFN6WbSwUk9IF3nnFQ6Hnl6rFlaeb40JUGxxK2levyrtCJdaAsjk1rKBlIgYHJd6pys8puU8j32ukPEJaHCzTIa1y7O09ZXToASikHC4cR3R6XlHEAPq9gZV6nvEfmdcl3rqKTkhDfS6xfkK7ePGe6jhqOgD9MxSamA851TQnYcKwHJqOsK4mTX3mnn23tFmxJ8JTl0MhIm6Rdujxegy+TXZNcRaIKLmk/S8/L7L366+ihhoNqddFNCC32YHiP8AQ8sRpqGDT2mTHvw9FUyHJ/mFCZqv1M24RVzxstBf2MD47eJKHT2FvXNk7i9/gyeljvbBaNMRmWE4TZPtc5uGJ1E9xqQ/rzZuqFRV4dKgv5D4Qarf5eEKEZeh0zgR1p2CkpP9lkm8CNubEDMD04S8mjUkzuKg2AnegoGk6pnmrLWvgLtalFWjQxiMvLtrNJhmkBMPiTCVosrgoqZJp/ifP9pR7vPP0x24R0VnuIKQDEdVfiioRjvAMUUi6GQGcZg1vlrleXWD3YUUx3luavywHi8bwCfxSqLMDBV8HMIkECISpQfuBPTB8qydxSTK+YPJDhJdYNsrMMFhoFqce5/lKgpHcR8fcLniZc2THKvlMptDvcuPdIdWpagSN8fBRxMTRL0GdgO/Hx0SrYpte4xIa5IWSNO+1cRSuZBIYRY2zT4QjTS7k1dVe5IPdz68+2/voryO5mUW9/Fqnv4MtLM8GEsUgqDEKCW1I4t6j3p2H0ESvguzWA5Y9ZAfx4mRfTXa4SVpM+Jy7C3eE/2CpddxL2JgG7rKOwdPPWYs5XoVBjNqU6MyeKiz6QLuBoSiHokRyBKSakBj7LCzndMG6Gx4DAKO4Wg3Kg/Aga0SqwTdISEqU0CK1N1LiIEtKhATMk68EpkER4mcOWwcGlVki6gI3gxK0V0AR1Ldfdgbg0d+x+MZqu5MucLAKtgbLQZO+YDDHOTvkt9o5tPJNek/EIccQK2tgYB3ZuU+t0ulaZBx4QntyZlv8TUb7FYypfQx6itVakgTxSg2GMBUgCur5kXtbXksUqStDxWjJCTOLKEWvr1lGKDlTxJnMhKh1NyGVxb6x/GQ1oiBwXF5wOrnD1Ct6ThQKgA940epOp4HqHs9EPbmxx9u/S5vJ3Yd8P+nzQk6M7X4fZOjNza+Q4L4IdaZH9aiuGPAtgZiR7qlyqvcyJkuP5wz4px9Ti7wm5lpMqhRKI/LHPPmvZqhB7FT0moZ2/lNjeKvDHZTD8IkRJt0sHjb7l30GXHwmayH/OUZRP9ZkW8qrlpbHmQgwKWzh8EoYiPE4MescnkIEvcFzmoE+5UMQTR8jcx4CbkdD4jw86/glRl8BnZC75/I2MRGoN4V6VOSqoNdxQP9Ynw3nS3lJCO4l2JCom4TT0gHnn5prstUy/PuYfWLry2iWOX3H2O9ArG+3KPq+9gfHFOywCHILB5L+rY5x7TUT0Nkl3DHWMxJiaQe7xf6DPTKV3C4mhDB//inQjiG5FVm9dAfHFgV5IGxjuIwFB0GIwQvt10vIGxz2s52b6v4XmxuI1U7uifDVoqNyXIsVUxk1sQXJkINzMWP1wBvOOzBI1Hm6gEywLXlEsEwyRVbkbz+zlEtPft/9au/8UGzytH1qC4xxcsRbqGjS6IA35oDQr9b2CBBitRjSVEfMxgvBaldqo/raF2gfZ2jKrFj8k4V1y7qaAs7hVaO3lvTklg+Sxw3Vs9e4Xyuephr5TRfe5bNj2Z6jf5iqL798wAWQwoKHKhwxa8Neg4CRK6D4AZyCM8QILNEdaVMCtD5PQuSHx26EnJq26H7QTJcH6uMbRGJZVmpMe5CaVgacnueWsTsv5KTrnpJpiA2fHCpOw1TfAiyQ0d0lZB/yLJHZCWKCaBh2BP3cCrgI47WtWweMZf2dFqEVmeNQOqbhJI4j5nqynrIlV9YE9WKllDx7lt1NdjfGc8PvlX4TnLob51AG7NBqR1z9Ox3FFtl2crJ6pDglTZ0TS+ZQoY9aTsL3dFdzyvIerucSOCgSQkvWa6wmCGN8mQhJOa2XBEWdw8SLKfp3Ygu6xB3K+NSd/NnNH6R5l/1K+qx7twrkDcBM/zWklrHNRT8Kuj870J1XKAuMPK6OFonG+WnsZYVXs5ttI+5Qxw1ZULROxUNsqEU/WNmVdshcVJyERXUI9epsSxdNSnPLXoWia7HU+menduuFWRB3UUP4A6mFuVoVVshAA1JEf4NvZvC+IzXmncGHbQUubfrZd3ONx77M8egjLCNd151++SDhqEJF/bPwuQkqIpZeknhAsXG1tkgTRy+ronbKFpmjwWsYetzpTpDakDyqKQXkBcAFcW7+AC4p7dtJ4iDciPkEEnQVzxV+Z+2LFw40Q2xoH30qxRjruvDAKX/gLiBiKI1ec8XovGdpHGPQ6glhWpPt4RbisQGK77D++aitpbfegWkqVx/4JCa2xBMqTsh5o+TOlVziXaZnS63nQHjchevcapOLnBgT+i9H45vN3CJ0iqUeRUnAmT3TAdEujEXpZ6mC2dbdLFiGRoFGoZ5XYfXcGwSLWUUWQNALJKUarsjFaLFZA8oVX1aYngqvR2k534OWd2rbJ6a3atuVlmnoO6OQMlVsXDsOonMs8ef0r1rE+wc8u3PPFl8FUeOM9AnAXJWU/MXDE4TMgzQLNbeO/TW7OhsULid04Qm/YeJQL8qyjMCu/eofVEIshSTT8hw8gWzZrYB+7clWerkZD6egeQPtx3Gk84cHDSy+OOoKHinJ+zSIc0stsoeA8lUFS9ElOIRIyR8Rygx2JI1V6gxxQF+YoL9OjqPRXqAjzWBkWUJNl18FgggVil9qM4BL/BQbggosfs7wmBF91cxcf7rMIFeGwBsgpV4iokD3nVkaSRKs80hELMP02jJMRg+ZHoLKcEwYjNClNymiRRxxVUwpw4D6KOuCh2eZ/HQVRnSpVx3rH/zRPnpw7JMYmEP7IeDZ7oLE48CdZk0Crrht4QOVthwG6lyHg9eRqPZ1BfjzGg+j0DgUBpAWcUqOwXthJ3Jy/Bg+PAgnjS8ZtQ4NZx3/Qum85M9p38ch0khViUkRUe02wdiUPx2ejpQqa4a6i+5F/J+JWk10fQ0dYVZUnCiEQTW7CNxL46mCspSnIsy4K32jjt67tx7GGzjW9wS5R5gSwIoFZaWRhYTnt1tJ/8BlACWFZfXszp5QCM8SxVMFkT2OqDvSBnQnN+MxE8bRZW9lU5czF2V+af0q64AD1iWRAL0Kk8rZxNErvD4bqykZsbqY8gkR5JKncEIi9xlVO/w54EHUbAPMJGLXlE9tRECZ5z/2cdrCpJ0IGEa7thsprRl6OlhmHzIsjK8EuzRukdTQke7JrAL2OEZTIDCyEs5VkTMNt6NflTzs1E0tKxmb5VwqK6u9qbhdJl6mHgETrq6Vs1lq+Oe6MGH8m/MFTa8Waq65yOawdYQFm1lc3jBHlcaYpkbGXIc6YemHpwk7D9oAZii4HVOU/sFk8lNLsXqU0JrNzi8+YET53QKt1rzEHJG7W3ITPtN7I1oT8ubMAQwjzXPYAXoKdQFljj+NX31/vEIFlqDBqzRc58hwXTvDLqSQ3OrYtZQz3l7OQH2i+S2k+BgR3qjfEjs5QCXP2aE1u93amEvbp/J04tlQauN0Tpl6ZyTrmNyOKb2CaS8uO4+YVmAXa7GjTuXd+lCV6wJa9R4c70kzp0NqTG63AeulD9O79x+ztHnvq88v49r+suliF6GXr0SUMv6tMqwRfx7Sqx06SPOYnbiIs3tV/Im6VQtW9jbz44CiwCqB69QHke6azPeZ4N1yK7spnSJrlO7sOVYwM2IamxCGWov4ysUsaeUWuaC+kbege8SVMmf1IH9v7d332jgjujJcxIqYp1HZ1VbErXRE58hoyu3I/ZIRaMIV6IPS0lKquph9SmUagOtQGs0d5zn3FGhpOWfKGBpjruSdhel6pY8DuPF11RApbKj+PaIQWiOATkN4pGSPxasVUa1+5i02o4eaXFOPznIU6qyBH9ddTkbejaaM9J52QPjnpdhbG6kkes37J397qD3cuc+NL1GzPy3pD4Nf7dH3OPs9pZGq6O+TZFa449c5Hr0FpRDuH3oVTf7bLQfj+hpewVVrf6sAktiS83seS1bznBl721kzfLWkLQ3Jyzlot0GNbNo6vDx6Is6mkPhHs2vIvszRj3yF6e5qC4ho0Y9lQ5mps4KmPV387rmUnkB8pbLtT4J+AD7ChLOncCFAFKxKbQzQXw8Vg4MoUhypKYb3ZjzKIutUgCnpmBD/YpV01wgen1VhHoSiYRK7fXGPsC95xHSMtEZMA9q9awysCnYdU9aboz/URUwC81SUUgJLyPa1USOUslwq0xPm/NFGsVE299eE+popYc9AqtEgWNE0RfeXCfQaVIOJI5sKP0rtWbqIowaJRexC5HhifWqNGWkIo8yAVVwy7x5NFq6DemqGGXcwv3sMvtKXbpRgrLevNZe6+29yHldWMzcZ8wkBD35awlJ52DmWKtnru6oFib55zhjuhWJrMdJ/bA1dZEaEdz7TgVRA0kNAbVVeD+myMGJ8XkDpF8bPfzgyYBY38APU0W+/fu3ZD8UB+WWxcXUlZtlVOy9AHmhkDqzbQJ+VSh7xgp1Pic4MstqypvZWhQY0OFRkQmlY1F8sQzRjq8BFrSEiNVLfX1LZl3w9a8Ub0QaQEfCKlGyDyXlp+rV4z0v4Yh+iXMo4s3G/bbSolidfNeM4F6UKkbmhIj9JUqzuIKEQK0i1nk85xvft3Q16LJnzUM7UHaNKRLEAxi42SWYNRyt0tCCXwCOykqYGfvWVtvtvkdN5dN3iZzskXaljfJFusUXBBkdo7d7jl/4lgyidJuOLJNvym7aZzKosYEnH0hByAa9EXzUReDVYnw7AqQ6hcjycoEy2JUptPTLc81nGMhiuH7401wvlHtnNMsDmrQshiS/3ZnetiZ3rFVqM97mXzq6LQ+Z+iOsSDHBjfGD2mT+LxMFA700dad4aI9r5a2gXUSsjtFmgqNsx8UgiiqLZT8nFgxzZYfSX+pe54mxIu34GnO+4gMOYgCCOeJ7SMSNVGP9jzy2YyXc9kNR1OPqF6kxCqN0K5D51kzxFnf3bBqCSEcSt7ttN6IoYxyjtLh3fLwCCPF92++x0g7z1JfbWL+iJ/yQghVWrs5uSPtlXlClbQ35+SOdBQm+KGfvUzFJemsT8mkLd/yxJcVwjuS7+Q+O9lVo5UYGHiQlhqIaHZrByAR2gFXs+upVtePXnTK1NeyBmfqNMeOzVIK94MsJrt6ps+tFHcnaAaqx7b+47qmaOlJXOS9fMnP+7cx7SV3JfkEQg0ip40LgyYz5oJ0jq0iWj2KGu1W7TphNUF3dWzzAmL5dxeBINka23PCCLW+DR/YmsQeR36SVj+Z6nTB6Ka4qNOd572OQCSpV/DNu8dLSekVJzOKYGCutUVxLBVTTaIovcp9TFFUpKWBoXUM9snPEVLMGkLaedbeq21+yc2F29sm8qa8KGJJh4Bp3+KZ4yY58YRPQvG6R8kEcxQHxjmrHW/5xDdHAjliMABekRp7Jlq08yB5sGAH+ckQGvWLKQ2VCVQF6B0caSSL38UoLvv4XIm7gzicfKC1ZjGEs2YfTNdtV4CcZynKzlhs0H+dpO2VGIwdOO5OE7fx75rxvZRzZBDXDq1XqnCiLOGZEInDNMnAKnZfMdTPkxPMrPFc3cMRD+pFiidaqUTQGZHVnRFTYo1hGhU1CihSroxGPol3/Glyk5jC4AFMI9JKnRA07GTgAamaT0NZul2WZHNpdkwdf3TP5xPV+K5x3GIrHMTT7ZoCChhbNZ8lSfSdZ+292uaX3Fy4zX2ytyvlM3BB9JHO3IStxDPONGrVp6xzQZYHmzJPW/5y1z8/CAgYD9wYPACC08Uok3LqyYHgwWjmMgB1bmxYD2nIKMMP/0Jl+kPyRR2VRdQNVAa2fqCyiCNbLSZMZFQWc5D0+ssSly0Gg024DFpTLbWtqyQZEAA0EYkPXBZRTay5JApCQApe7ZOiLNmCSthW/Hm1KD/p0y/S1HVFhNdTQiQYv3JOq4xXMDW+5pa8A2+oIn8KmenxXXl9qKk+MhJlg2I+Lkobgypl/MGE3DGlW+rQJ03n9HJSRyWRlFN3FVmDSSCPPeaaxacwacyWLRJM2nnW1pttfsfNZdvcJZubcu8IqOeN013q8Z7sJG8yZ4RU31VpbOCOs1Sd8zz9T4wFEV3lUV4V2+QtbaCESOCoKIWee5i8H6fe7UZiBehIWqK/2xWcIFeqhJAOHqEpOdkO5ELZGM3mPHLZrUmfF0DufMrlWWlu3GPq82axDuSA57gY7NVWQG6l6VUZyAHTQJbO6UAupCp1lpZVuVPu4epArgSNJVlWoqQr3kBaVS4XGfWyEgmVM0Ik9ikDJPSgURuOSOKbYp1n0tCU8zNU1RgDP92GqGUyWAMXjHQQx+s3JIlox5WlIa0ujtUGrcHUP0vRBJPOq5AkyfAJVikErc1H7b3Y5nfcXLbNXbK3JzdPwOaB2zncm45E9ltc09P9ZKNRdZJf5l8pxwESCzgSbJJRnTM0dgqBo/yuG35W54xSS2kM485aVaaK0BRBuiFQiVTlRIZ/1ZBXuoBxDunhRSI0HA3aQleT1xvjv5o0/fRlKm81W6gyBKxBm7NVGAMCqVIWOe5gzmP1OWjjaxnMLcQ+Ilc9j+Lv220n2InMNVmGhmTATCbnTW5NZRWRKJdcMaNyvppAIoUsGSGxxy7S4DkWrOr/SNsQntJSLLN81CBn1o00V0cRRZweDXI5fSklwgEThHK/kKUNZaX8et5319HBoakFbDxq78U2v+Pmsm3ukr09uXkCto7b5tne8ySbfmvPS2765M0QIEecQWsBOznEcYIrnNXEzbFoE/v8iDi075ti5s3wvQ/Ds3DoKDoa/Tc0Lu5Ib775fdExsuFZs2tUHZOU6Og41ecNCaZuB5O0LQnsK5iK1cOgDcpimGrNQb+Am75J8ygro03vNZJVZrTpk7uX5r/IN9K8AomC1KFm9s8Ffz+gJnA2NHmWGWmq/f4kKiJ3wZE0iMwLJ4kPmfNESh1qDZD0NuQcF+tmqBgOQ1jqhyNs6FioKSduGx8pFzEVB+3tEs6ZtSGsWsYFEezAx2VpEUjPYDj68hz5lfJYOUF80t57bX7GvUXb3CKbO3Jr+2+eNfloXySrnnuSTb+16SY3vfJmEJBjDnPi9CBXgbsnxtQheApmYgg/+tZFsNGBNI56s8YAZdSEODxoxi86JBmHeWA1mkPiZhzm4Zo2qrBeg5qDqAaQ0WmTOhhOVyOXOiG4b8yFm0LsFSg++zFTAmvGmpT1bf2PUUTFGaeYB3E1Bjk1S4M2wwoWmwOkLmBxjBJVLa2K8KKE8ZyD1TQEp2q6KMcztRzLmk3s2NRuQmwAlpnyXNhWmWN0z1cro1R1URN/3BOlokYM0lntH6XusuFE8/NqcT+KWpaR0lxZItjPVd/SX89tNHQWLeIi2Bxj5d3zDkvxWXuvtvkl99Ztc5dsbcnN/b932vaO9qYj2XNbm05y0ydvhoDNiCMHOJzpp8dTGCEoh++Rtjp7kdZgWNAg+xkZRs6FHjjONtuVyckwtg1R+WMkOVF8TJJ87lA6uOddo92sZMhLamYe5PSGyqomz9oN64FFaSe9XkBBm9FCqmFhIPWKA9I/DKGxXJr//oGEk4/3c00nuwwtWotxYolxcEjx6WCNGQarml5Hw8HxC05wW/UVibJJ3B0kq05Ql4/cUcmeWu0w4JYblULHnTNqiZhZz2p+kbnZKlZkGFC0Pg9kuRdxwCEz8YtUI57bPPodURptSU0Nox2+PAeY0iTovUftvdjmd9xbta0dsrkdN3f/5mHbPNubrmTTc+35yU2vLEcBNhODDncGq0GOWjzkYDxuRgAWh1IHNM8mQ1EwAbhp9UzJ+Bg/JLitbhVpDGKSrM7jFAZ7Uel+aG9xj0/uopJV6lYRe3Sroubehha+NDKTk3QHqBCaLbMX2LlUAG1O0+tA8Dxq1yJK7ECttqezuKYstCyP31Gwj9qYv7TEwYvBbYUNGwo7VmkoKKWEZe14ovPK/EP2LdG+kLQVvdaUYrhlyVrdibMhkV3VtgkOKCotkGlHYk2Z8ipqepDoWyrmQ/hQRNE2IsEVccQVJ96685Ra3JhPWLszfYwUh6JDfM5dFJ+192p7H3Jr0fY2yN5u3Nv6mwdt81zLfmTGYJLfusFgoqrfnluWowBryKhRh3VsWwINmTGaLBrpwUB254nEC1G5CHG19Qe4rHSt1EB2yHdMygW3W6UKCnZSK0DLMIpicBaVKY6tYObTkNqdrc5C0IOv6AVBxmFXSCVFyQ8h5F6NfixklgjwVSXf1u1A28jq5F3j5wH3JMG7jmcP5BlrwbTb4cwSU6QorGCwOqWQyBhy7ZgoFbIIPoPZqErxEMFB5hwyTUHVAWFnqza6ck3FakC4gpaQKFJ5gBzxrErkrMRDZTQ1VUelPhW4IeAYUlH+eaG7X3o1tEdV3e4zonuO9rLbENsVn7X1ZltfcW/FNveHvB9vKp8if1M9bjdQSmy/3nMmm75L9pVstuWa5UjQ4xPIMYuBZ0h1QFwdlwJ3Hk1eQhC0X1qALqiPCbblXsGlm3FzsyHHgT3KLcIQ04ESFdWeYXcWDRwwUSkqdbOAyniHkj3rZqnEx4J63ew8wmM0tigsxW52kkQx21rYKgGJZTS1BMkKS7mabjBdJobei6W+kglvH8fzOVJ4JxiMCGXSRLcqcHKcQmwjbL+aopfJLrXn86gJpYuDHSnFrc46YrAd7ZGzbZV1Ngs2d7BZlXvjJLDFkpqpJeTTrbbWOQtM13G13YMjnUrd49SGWpDl8K/m5yhJpAI9xkNZonpwpm3MhXrc/txeR9nI6onP2nmxvY+4uWSbO2RzQ27uf/m8zWBIOt9spnkTVs9TvRc+TXWWZLXlmcUo0N8DY457txG/Y340B1OEwnbtjA2DUdeJZPUV6X5CSqNmmdmuHdDJIo3zHmY46SsraiLtWybos2FaocE3D2NiB6AsWTLzBw75Uponh1nDIV9KUaeNHiRoe2lREZFp/oXyg/FQonc3y0e4n99+YVe8ICM4v11FpZtSlNxcv3uA8qfrVwNlo/QbcZXaVvhpoKO1kCuPZJcK2GWnSCTShWzgyty0G9L5ZiWO4+YbktrqTFcdVayGLi3mRIIp2Uz3D0uid1oEvEgkuzUqre4EWpmYnLrF+5oeRnGn2HMtywprawS6GTJr1dQZ+Wp5uRnBWoBtuoQgZOvX4udUxPYmxz0bEyU+Sn2xG8Am5jf3lm1vk6hbcglqNN7j1lETjzUZ6T6kodmOy9pzkKo3pmSE6vt7YAbPcz4rHUwaREK/sqkpCrM00Grgz+ufl/PKqlhzexe/ML+niN75bjKSCD5oNDZye26UlMZGNBLngrDRtTzPjVFLUnoWrLr3cEoEY6ta3NNnDaxp9Z77lVWp9bFVB5pRiuhkZQ45ibaVrWy1etZgPqenX15WmmarFJXCaJMGp99Zxfz0mJhTstYbymq1nPK/kibyerk0xb/JqJSnNlbP7/rnFfuKtPgUFqlvvcJWcXhtZeUM1ztXRYWAJd//xB1akB5EQX8jDdo2Kuttg3vZNjp72ka7+cYW144SHXb51JaFkdKL3TYG9mk+b7njFu41mka2J8da8UiAu3RGf8FQQ8S0Zbfz57qonbbEAbf1PabjNm2Z+Wd+TdHc5i0bP+5oTslbJjbzz/OWoyyQnuct6xCUao/zlt2snIvMYt5ymGGTjJS3HI7Cxfu85WyW3H3acrYKWD83s5aF7GCmz2L4SSA7d24UW6QtcZ9kVMm30pYMT7pd9VLeMpMZSM4u8pZsF32U8paF7BzeQc28JX6W9GqY77TurvS8hMIai6tyJTtp4DwDj7o1THWYScPFpp+pTc+YzDRxaL57VE1acFo7USOGH1e01l9e8iL2gUx2IqOQlzyLpWZehLiRTdzAjTuPyVvV87JFySxbbTtlpy28bokV6bv+vLz6ETufFf08n7du3RpNuOeqHjjGgmaqHyYz0e1jYzGFp77HDLPIUS0cTaFH1kVQW7AjG5md2+QX7EhMN9FHWbEjG9kB8VBkR9LSqezI0c1Fwt8KO5L2l8qO/GgCe8yOpCNgsSMvgnzKEj2yrIL8gh7JQb41d0+PnH4mOhNnT7Phx/l2T5Cc3o4c3oIh2cguHgJDcgLYjuY7mQN/E5l5VGJoGr6m2KGSJIdeU9M4km0BfhakJviZRRyMNj1OHPwxgSZRN3r6naIAINtlUdiFf2cWu3z5u6StTpi4w4/0W4Xijayg3yu177zU1ufLWx3ZeUvyp2wJUT7Y9WkBdUUa8oMzHWyIrJGsH7grv0DWC+9YEFpLzngJyG3PHx2F3xwRE0ZlSsuAuAUShdm4T1KCEcP9orW6st3RotJaXTlu1yoNBFygBLOzeoqj/bO7et9aPcGE8HKpPO6tJnQh9lZ/gJB231s9m/nwvLWa9qXaWt3NYm5SazXbwQCnxZDkskIyVm/1BZKBYQNWb/UEEdoLx41HLR2D7svucENY+KHrJDXU4X4ex+UQ+vfugJPWLjhwE+hsiBMfvpSA0xSsXsjJTpx8OyW39ibk5b3xKnlPKTvtyR+mLS2cuNNRHfb6bLY4gDvd1FsP2nqpnc+3tVB7m2JvB6rbnXycerjIrGxNsVY9x4R0VT/lF0h3IThBSFdywvQbZZcfVhmLkkw4nhzZZex4yUYKtLhFpqMDO6swUTPZuVgUdUWyAzxiqiteZEhSCbfyihf5mDOWN/UVp6JS/w/uXmBxShshilEVFofCYYj3CoscrvFapOorjn/eZUlecYV9FuqKbOacuxdXnOzogmmJK05JnPwqSRiaxF+FnIPX1G9KeteKnwkr8l1d1FUc83Pb87HO7JsXqoq4BAfOrdLEyaZ0mCaFNuAZqNiKgop5b8xy3hrRl7aGraQtAe24o6UYthR5/Gan9lZfx46M4taD9t5q6wvurNXWrhA3IDlFcbNTOFNPFj9MPMcUc/e8xp6L2nSIqvul76g6+4l7OPQGAdIZQByn6jEIGSO1JXLY8e43uR2ugpXfj+4HbbYKp2DS0R6PVhlDSrDQXH18DF/MySpT6qa+M2aPJquMuYJBGKwym7mWH49V+cjG1qdTVRjxWENVZjOPheamkN9Hj1bJioo7LVuAUU2LkSqekFKO+fFEFXYM4kCV7oUCKikoSvoDKIHUrDhNJZcXljUOqRE5J9RHE4epcBQQZ6nkA0U6xFEqqaBimThJJe0Nak57M/fizhiVsKWoHfaUGb0oyDJLT0ott2S2M0Jl60Gbr6V+RAIy4oqRlbY56BeqG5GepW57NhMP2YwipSPNZqIDoci7565035gRe0qOmN5MdfuU4xsg9ZQR6S/6dajdjabNmAB8nhXSv48ihUO4Z2gnGSKJnswCanBb8brGRalrMdJ5sjune1YTnVclMnOg8wyZ0hHvxzmvMZM5zHmCI6lvgXY/zHldkDNHOa+zS+YgZzajZLw1xnlOSkVsRpH4wRPWsodA0oofQHRUZzhzXUMc4czXUmue5lSjzDyHU5rg3J2B989FuNnHWqNJuWMmOxhYKM5v5iqWOL6Zchfi9Gaqs4nDm2NDNQpxdnPcGd0c9qbxha3JLH5LU3mq2i6ojdgfLQv8LACh1EUrP6guasSL1wqLzOBCKL3YtWVtxfZ2h7YRGXpubXrxgDFi1Q4z7SfVc3AmUnNTXMnWnWJC4Cm5YP6NosNnJt9ASBlgXTVQNSrM9L8YkSPnjOfh0JIyhrLA+JFs9I5i4Xa0TCeobR6paeQg0qNuVcv7HAXybi1oFTLMu4WNtM/IaCq37HKAavOYVq1kRhrOA+8Lq/zGPNQ6UY/6UBJo/UZEQ0u8osnXt3Mo7nEZNUecZ+Tf/8pD/OL7lUgBZw5F+IPUOzfwC0yF7te9pqSQ+70ZpGBFwcGUcP5LP0iKlUd1MXHKSayowZiMPqwl8OmuRWmxjwf2z4kgK+xNPfZbA/D85iyVPY3tHRk5RT6EnOqWWqP6SmSmfkBO1WmrxSm3ra2xtw93tvze6VKPMvkb0W/QeqlOakZmOz5R9cBc7BX9PbdLEN1m9MIaXzIteDMj19SUkR/dzqPQSTN2cQkrPFFfIUvPG7kT4J2VlhRRAqyCD4aXVybmdijiItYb00ZOpD+9eg2KuJzvK47TzyRGwWFXmDOBkdIAMgWjuQazdbw73atITCT33h43Ong3TCSrR5XxQd9U2Plg9rqkRRam326S0knbb4kwM2/BfSgL7tO3NqJ7FAPK1aJGM7r0aOTV2cih+ouImIgnJOalwtbw+inrI5Lc/NZIrInXpU1K+LIjh7uhKag9hhNF+rCV8/bd+4DqajH4EfcGZ6V2NuLept85X3tHWXQbjH1EJ7Xk1C08IjxL9L9UnxSdPWkXjxB0kg4Zgc3YGikQgDmToQcXzFDSSpnsXIXqnwX6q18Bn/xqTqzptAMAmrUGddUJVqqle1cdAZjsgQyWjcZ7xi8FVd6OrIwHodTgUIdTBp6NwvA5BdbE5oW+O3LS1OHqCvYMVKvUCLLHqdJqz2c+3toyn6kppW2Nc8gddSnTe/u9yCO1KyiF4pRwII/ICKOk+kKmuBEhCYTe+4pI8mLdyR74vCKp77BDd0HCSzk8n58RCkqxiPksqrmIRUPfkHIrss/81uz3qbimDgjemiz1ZWeAgCQdS5Fq4zF7b7T3+dS1mvNS0s5Yl/4W5a5zXFP3PNfxxBO21EIxDjM57weOIyJfTfJTy0pjkMrrqgOeOW6Su0e21Ih5JxwyImkVmi+JP/ZB476GqzXZTF5zOMLELUFq3GgzVFQB+tukgFki5VpHsMBsM5xgiH/5COUxp5DZmWU4vrUiFpWG8iSgnqZMqWQuZL93HOUxl8VpfVbM1+wXvqJMKUu+LwHUlhU+F5NK/TsbdgsKCs5EE9ns6H4W0meN0ERD4ZwiZV+QmyumlUJEtRCtEsf0YZHuxFkAkTjPHGexe9JtzSNnwrc49FYcR7TEIEqWYus5W6+09/n21mpvY+ztQnHHM2F+53SpJ5mQlew2zsFlz0WpDnGq3inOl9NDsquHQQXI1s6jHGwkJliBMJ3nBuSXU0a6dPxRg6TqNdkdGXS2Dq+IZ+dBgwbOUk1Vy2hQlkfq3WPk0r+tlHcZHCKQYzgUglQeKVUcXWox5isBlxYgO1Sbl+o/qUh6r5XwztEE9tFk1+9XTZBxmPMgMGPYvWLQqkZffeJnBa4plcLYcHSdNTmAoUs5D5011cCWgMf3G7KQT4oeRzeI4Co0lOnVLlyjSAVybKJgRPCovSESnXyFXk2RTe4jEvPFvkTXkGkmilO4hAl4MQ3ltuZyfxEHMC6AkjZ+Z+s5e++09wH3Vmtva2xtw70tv3e+1MPM+R3RdSzTQpaboti55RH3vK/u6/OqqNJX4ihCiYrIJOWbKL+QdPFeamyj4s/gTYPMgdQ1OYGX0iG5k8BLjDgtKUsQCyj2g6ZtZSbcgrxiSaBO8R1rOCMVJY2MTAXqoCZhaXoepmu6h7WaEhvla0qFMe/RyEUhx54SnfbIzrriynQ8aeyUUla9TrZSViXQ4xHR1biBepx1X8sEe5KT9FbTqrVqkcLKC3LOovyWF8Umiwkw54dAEcOiTt7kh6z2FDbzwOe3OpxvSlRREyRh2owoJEvNXFHTrtoaO70xa3VvvPXOK+19vr212toXe3twb8OLh4tTUfJJboumO9Fv7HkpzSPSHtxyvrqnR5FPxhKLoT6EJZKHdM2hENIGlkipKGKdlbBE8qjslDQsEZE07SVt0I4JWgEMYiWxCmdCsr8f9TgSK4QlHNqVonDJU18ETNhYqB8FFeK3qZz3A3o8XeIDDsWSeqr6VQ262sTGde+Q2i1KILmClCMxr+E8Fvs0Kd8vFXNY2kCILxHPqnahlCbicjjbeM7WK6mfj07o3mLt7Yy9bbi354ND2TWNARoiDjNZ8BkKUkkSCs1GoT0nHjgYbzFJNWEcPM/OVaMTxEGr0XSKg6KXwjvPCE3NKyKHKGtD/Urj9m/Mw8ieYhNQP8ftX7mbUVvVuP17RV471R4+nNKlPV2QEzJbj0MaKYW9ZqPzKCtzrqm1zVSo6TfntmrLFOVRXENqq5g959ZHUW65+7ujPY9plVhOsUiXnwNHEKen3RPWKHSOFhvP2VstvlxoTd3cOKq1QwUPgpsLuhekvbjfdEHIyVgfP88LWpWnwqJPdSFpHtEbn+emmz5y5Y0PiyKGDYhEvhpiFO76F6ZEdgdqx3qhLZOzgHb5OS0oWx8zBJzwIzvgr1UQnGVflQYNBWYPFCPpmNqC7VWqxcgsaZWqERnnLkIZ2VSiv5F8XIw0wlZVvJUsBmSCWcD+6cWkdWjDSqjyrKlKdFR3lhAx+zFv1CKt/gJ2JZjeWMiSVUxvwLA5uz6+4OfYiWLMbhw4t3iwUQVUN5YW+ZhJue6Ood4NxauNRqdIN7sz+cIU7uVrK6favK3aE6iwizQdqfRPPYiLsbIJL0LSu9E2ph5ETbiuh+xzi5Oo38IhW+t87leahNcFa8RHrXZqz66S4IlpMIlw1UQeFg2PhxWhUGUpdYAUEPcb298nqsnkICkHhEqNPK4qTCdgjQ1cHKPkfCJ6uoj9+M2Q0y0ZsRaMNrIpRFB57VjrrC+/KKY1vPfGQ0p/IUJrOB+z9fuVEwAQVu4+cEwVrodpHC+wS0ZZDBsHvHtXg29FnbCIzRjB1s4EO25+sbn1kEsM76h0SzWri5aZRc214C6BoQd2mhQK0dRps7iHQqWW6gTfRyNdHNNI196SkzCaIWFuNaFg8KIzE5at4Egjq5OCU0YHiqVoxMIwotJjTZAQX7hqVlIcT2l+S8x+jkkIVbp+YY7DOtsJfeuQXoTZY9bwlBTNivdiYlnEijdFjh77jbOWF80GMjPCD2Uip7BPkfdBEX/R3YMJKj9oVVVpTkbPxZR5WzsGqBjqz5xuANJHWTLtV0tA/u5Absrhw1OC/iLx1mzstFBVLPa94aNb3gAlbqEpN8SsLXeXVgcnWNdmv9B5sylFcLh9gYEfC9J2I48cc7jviWPeE2XEzJb+6biNS5dGYIoYAbS3Az8pf0ncxw41IyxxhFV1fyhe1fZY781OkVTChOGIShsBOHJik1ptk8sbkdhDgBRUk003+Z1U2n0he0lcNTP+k9vxtSg5LbhFEd91JaTmccFywyqIgXuCJz7WeXbkImOHU+77Fg6+KdWTiFkm7Kr1wqinfjLTGflbfMspyR25EUapN4pvhcki+SPi5TC+B2t99jRb2ioRd8h5cPziwkzJUuA+jpkChyJUQ5pJi1xwTItTrZFr+6mO8X4I7ZIstagAohcWX2zphfO7+HgnU5B41oel9oOZ6r5m7jRJtvTAkRU/nN7Ko/eM6Iwn7TxnYsVGinjUfGxK5XaVBV61xlFu5Ouh/AQ9WTGLMJN39O8ZpAC40hDbYcVJz4vdv6CDeZslYdbOJ1fXqhdKQwxCztyKMXelFCkJ7wrqg4Yg5GHSaPn7/JG5vS8pN5c8n84C4OPVjKdBimNs5EMa0FgXyiWL2hxurR4cz8wFOxGPGMvjKMJk0jUxE4l+vH/IZNSFgEfAO3JQ7KLwJSsvdzQSmHWhTbyYsQ6yYX1P+rNLiG8PcYFiEh5Sh3Obvq7+zdPG3kKpWys1BVc8X0AAP73lXW4A2rgon+FgNPmyxdl1uW5WjeRBWa12fJ/1dd2XynljoOQ1FOQ7aD0p2fUNeTgBsVIrQDnMkRN5hUfyQP5JQboYbobY/yFMj6DaUPmW0L9LhYV+7zw1cOdve+0uqRVGIjScf2Ywzil6BV/f+ZrvdtbM+Q7v/ue//391qzCge78BAA==""",
    "month11": """...""",
    "month12": """...""",
}

def _decode(b64_str):
    return json.loads(gzip.decompress(base64.b64decode(b64_str.strip())))

# ── NIC 빙산 로드 ─────────────────────────────────────────────
nic_icebergs = []
nic_data = _decode(_BERG_B64)
for b in nic_data.get('bergs', []):
    nic_icebergs.append(Iceberg(
        lat=b['lat'], lon=b['lon'],
        length_m=b.get('length_m', 5000),
        width_m=b.get('width_m', 3000),
    ))
print(f'NIC 빙산: {len(nic_icebergs)}개')

# ── Copernicus SAR 빙산 로드 ──────────────────────────────────
cop_icebergs = []
cop_data = _decode(_COP_B64)
for b in cop_data.get('icebergs', []):
    cop_icebergs.append(Iceberg(
        lat=b['lat'], lon=b['lon'],
        length_m=b.get('length_m', 300),
        width_m=b.get('width_m', 150),
    ))
print(f'Copernicus 빙산: {len(cop_icebergs)}개')

all_real_icebergs = nic_icebergs + cop_icebergs
print(f'총 실제 빙산: {len(all_real_icebergs)}개')

# ── NSIDC 해빙 농도 로드 (현재 월 우선) ──────────────────────
from modules.config import ROUTE_WAYPOINTS
current_month = datetime.datetime.now().month
month_key = f'month{current_month:02d}'
if month_key not in _ICE_MONTHS_B64:
    month_key = 'month04'

ice_data = _decode(_ICE_MONTHS_B64[month_key])
cells = ice_data.get('cells', [])
print(f'NSIDC 해빙: {len(cells)}개 격자 (월: {month_key})')

# 항로별 평균 해빙 농도 계산
ice_conc_by_region = {}
for route, wps in ROUTE_WAYPOINTS.items():
    lats = [wp[0] for wp in wps]
    lons = [wp[1] for wp in wps]
    lat_min, lat_max = min(lats)-3, max(lats)+3
    lon_min, lon_max = min(lons)-5, max(lons)+5
    relevant = [c['concentration'] for c in cells
                if lat_min <= c.get('lat', 0) <= lat_max
                and lon_min <= c.get('lon', 0) <= lon_max]
    ice_conc_by_region[route] = sum(relevant)/len(relevant) if relevant else 0.4

print('항로별 평균 해빙 농도:')
for route, conc in ice_conc_by_region.items():
    print(f'  {route}: {conc:.3f}')

# Drive에 캐시 저장 (선택사항)
for fname, b64 in [('realBergData_latest.json', _BERG_B64), ('copernicus_icebergs.json', _COP_B64)]:
    out_path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(out_path):
        with open(out_path, 'wb') as f:
            f.write(gzip.decompress(base64.b64decode(b64.strip())))
for mk, b64 in _ICE_MONTHS_B64.items():
    out_path = f'{DATA_DIR}/realIceData_{mk}.json'
    if not os.path.exists(out_path):
        with open(out_path, 'wb') as f:
            f.write(gzip.decompress(base64.b64decode(b64.strip())))
print('Drive 캐시 저장 완료')


In [ ]:
# ── CELL 10: Drive에서 학습된 모델 로드 ──────────────────────
import os, json
from stable_baselines3 import SAC

DRIVE_BASE = '/content/drive/MyDrive/arctic_rl'
models_dir = f'{DRIVE_BASE}/models'

# 우선순위: iterative > train 결과 모델
def find_best_model(route, ice_class, ship_type):
    key = f'{route}_{ice_class}_{ship_type}'
    # iterative 결과 먼저
    for suffix in ['_iterative_final', '_iter3', '_iter2', '_iter1', '_final']:
        path = f'{models_dir}/{key}{suffix}.zip'
        if os.path.exists(path):
            return path
    # 일반 학습 결과
    for suffix in ['_best', '']:
        path = f'{models_dir}/{key}{suffix}.zip'
        if os.path.exists(path):
            return path
    return None

# 사용 가능한 모델 목록 확인
if os.path.exists(models_dir):
    model_files = [f for f in os.listdir(models_dir) if f.endswith('.zip')]
    print(f'Drive에서 발견한 모델: {len(model_files)}개')
    for mf in model_files[:10]:
        print(f'  {mf}')
    if len(model_files) > 10:
        print(f'  ... 외 {len(model_files)-10}개')
else:
    print('모델 디렉토리 없음. colab_train.ipynb 먼저 실행 필요.')

# iterative 결과 JSON 확인
iter_results_path = f'{DRIVE_BASE}/results/iterative_results.json'
if os.path.exists(iter_results_path):
    with open(iter_results_path) as f:
        iter_results = json.load(f)
    print(f'Iterative 결과 로드: {len(iter_results)}개 조합')
else:
    iter_results = {}
    print('iterative_results.json 없음. train 결과만 사용.')

In [ ]:
# ── CELL 11: 실데이터 환경에서 평가 함수 ────────────────────
import numpy as np
from modules.rl_environment import IcebergAvoidanceEnv, Iceberg
from modules.rl_reward import RewardWeights
from stable_baselines3 import SAC

def evaluate_on_real_ice(
    model_path: str,
    route: str,
    ice_class: str,
    ship_type: str,
    real_icebergs: list,
    ice_concentration: float,
    n_episodes: int = 20,
) -> dict:
    """
    실제 빙하 데이터 환경에서 학습된 모델 평가.
    Returns: {success_rate, collision_rate, mean_reward, mean_steps}
    """
    env = IcebergAvoidanceEnv(
        route=route, ice_class=ice_class, ship_type=ship_type,
        difficulty='hard',
        real_icebergs=real_icebergs if real_icebergs else None,
        real_ice_concentration=ice_concentration,
    )
    model = SAC.load(model_path, env=env)

    successes, collisions, rewards, steps_list = [], [], [], []
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = 0.0
        ep_steps = 0
        ep_collision = False
        ep_success = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            ep_steps += 1
            if info.get('collision'): ep_collision = True
            if info.get('success'):  ep_success = True
            done = terminated or truncated
        successes.append(ep_success)
        collisions.append(ep_collision)
        rewards.append(ep_reward)
        steps_list.append(ep_steps)

    return {
        'success_rate':   np.mean(successes),
        'collision_rate': np.mean(collisions),
        'mean_reward':    np.mean(rewards),
        'mean_steps':     np.mean(steps_list),
        'n_episodes':     n_episodes,
    }

print('평가 함수 정의 완료')

In [ ]:
# ── CELL 12: RewardAdjuster + 반복 Fine-tuning 함수 ────────
import time, dataclasses, math
import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from modules.rl_environment import IcebergAvoidanceEnv
from modules.rl_reward import RewardWeights

WEIGHT_BOUNDS = {
    "collision":         (-1500.0, -50.0),
    "proximity":         (-30.0,   -0.5),
    "danger_zone":       (-50.0,   -1.0),
    "route_deviation":   (-2.0,    -0.01),
    "progress":          (0.5,     100.0),
    "smoothness":        (-0.5,    -0.01),
    "fuel":              (-0.2,    -0.005),
    "ice_concentration": (-5.0,    -0.1),
    "episode_success":   (100.0,   2000.0),
}
def _clamp(v, lo, hi): return max(lo, min(hi, v))
def _apply_bounds(w):
    d = dataclasses.asdict(w)
    for k,(lo,hi) in WEIGHT_BOUNDS.items():
        if k in d: d[k] = _clamp(d[k], lo, hi)
    return RewardWeights(**d)

def adjust_weights(weights, metrics):
    d = dataclasses.asdict(weights)
    cr = metrics.get("collision_rate", 1.0)
    sr = metrics.get("success_rate", 0.0)
    if cr > 0.20:
        d["collision"] *= 2.0; d["proximity"] *= 1.8; d["danger_zone"] *= 2.0
    elif cr > 0.10:
        d["collision"] *= 1.6; d["proximity"] *= 1.4; d["danger_zone"] *= 1.5
    if sr < 0.60:
        d["episode_success"] *= 1.8; d["progress"] *= 1.5
    elif sr < 0.70:
        d["episode_success"] *= 1.3; d["progress"] *= 1.2
    return _apply_bounds(RewardWeights(**d))

def check_plateau(history, field, threshold=0.02, window=3):
    if len(history) < window + 1: return False
    imps = [abs(history[-(i+1)].get(field,0) - history[-(i+2)].get(field,0)) for i in range(window)]
    return all(imp < threshold for imp in imps)

def evaluate_on_real_ice(model_path, route, ice_class, ship_type,
                          real_icebergs, ice_concentration, n_episodes=20, weights=None):
    env = IcebergAvoidanceEnv(
        route=route, ice_class=ice_class, ship_type=ship_type,
        difficulty="hard",
        real_icebergs=real_icebergs if real_icebergs else None,
        real_ice_concentration=ice_concentration,
        reward_weights=weights,
    )
    model = SAC.load(model_path, env=env)
    successes, collisions, rewards = [], [], []
    for _ in range(n_episodes):
        obs, _ = env.reset(); done = False
        ep_reward = 0.0; ep_col = False; ep_suc = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            if info.get("collision"): ep_col = True
            if info.get("success"):  ep_suc = True
            done = terminated or truncated
        successes.append(ep_suc); collisions.append(ep_col); rewards.append(ep_reward)
    return {
        "success_rate":   float(np.mean(successes)),
        "collision_rate": float(np.mean(collisions)),
        "mean_reward":    float(np.mean(rewards)),
        "n_episodes":     n_episodes,
    }

def finetune_on_real_ice(model_path, route, ice_class, ship_type,
                          real_icebergs, ice_concentration, save_path,
                          timesteps=100_000, weights=None):
    def make_env():
        return IcebergAvoidanceEnv(
            route=route, ice_class=ice_class, ship_type=ship_type,
            difficulty="hard",
            real_icebergs=real_icebergs if real_icebergs else None,
            real_ice_concentration=ice_concentration,
            reward_weights=weights,
        )
    env = make_vec_env(make_env, n_envs=4)
    model = SAC.load(model_path, env=env)
    model.set_env(env)
    t0 = time.time()
    model.learn(total_timesteps=timesteps, reset_num_timesteps=False)
    model.save(save_path)
    fps = timesteps / max(time.time()-t0, 1)
    print(f"  Fine-tune: {timesteps:,} steps, {fps:.0f} fps -> {save_path}")
    return save_path

print("함수 정의 완료")


In [ ]:
# ── CELL 13: 설정 ────────────────────────────────────────────
ROUTES      = ['NSR', 'NWP', 'TSR']
ICE_CLASSES = ['PC7', 'PC6', 'PC5', 'PC4', 'PC3', 'IA Super', 'IA']
SHIP_TYPES  = ['bulk', 'tanker', 'container', 'lng']

MAX_ITERATIONS   = 3         # 최대 반복 횟수 (수렴 시 조기 종료)
FINETUNE_STEPS   = 100_000   # 회당 fine-tuning 스텝
N_EVAL_EPISODES  = 20        # 평가 에피소드 수
TARGET_SUCCESS   = 0.70      # 수렴 기준: 성공률
TARGET_COLLISION = 0.15      # 수렴 기준: 충돌률

DRIVE_BASE   = '/content/drive/MyDrive/arctic_rl'
RESULTS_DIR  = f'{DRIVE_BASE}/results'
MODELS_DIR   = f'{DRIVE_BASE}/models'
REALICE_FILE = f'{RESULTS_DIR}/realice_results.json'

import os, json
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

if os.path.exists(REALICE_FILE):
    with open(REALICE_FILE) as f:
        realice_results = json.load(f)
    print(f'기존 결과 로드: {len(realice_results)}개 조합')
else:
    realice_results = {}
    print('새로 시작')

total_combos = len(ROUTES) * len(ICE_CLASSES) * len(SHIP_TYPES)
print(f'총 {total_combos}개 조합 | 최대 {MAX_ITERATIONS}회 반복 | 수렴: success>={TARGET_SUCCESS}, collision<={TARGET_COLLISION}')


In [ ]:
# ── CELL 14: 메인 루프 - 반복 Fine-tuning + 보상 자동조정 ──
import json, os, time

total_combinations = len(ROUTES) * len(ICE_CLASSES) * len(SHIP_TYPES)
done_count = 0; skip_count = 0; fail_count = 0
LOG_INTERVAL = 600
last_log_time = time.time()

print(f'[{time.strftime("%H:%M")}] 실데이터 반복 Fine-tuning 시작 | 총 {total_combinations}개 조합')

for route in ROUTES:
    ice_conc = ice_conc_by_region.get(route, 0.4)

    for ice_class in ICE_CLASSES:
        for ship_type in SHIP_TYPES:
            key = f'{route}_{ice_class}_{ship_type}'

            if key in realice_results and realice_results[key].get('done'):
                skip_count += 1
                continue

            model_path = find_best_model(route, ice_class, ship_type)
            if model_path is None:
                print(f'  [{key}] 모델 없음 - 스킵')
                fail_count += 1
                continue

            try:
                current_weights = RewardWeights()
                metrics_history = []
                converged = False
                ft_model_path = model_path

                for iteration in range(1, MAX_ITERATIONS + 1):
                    print(f'  [{key}] 반복 {iteration}/{MAX_ITERATIONS}')

                    # 평가
                    eval_result = evaluate_on_real_ice(
                        model_path=ft_model_path,
                        route=route, ice_class=ice_class, ship_type=ship_type,
                        real_icebergs=all_real_icebergs,
                        ice_concentration=ice_conc,
                        n_episodes=N_EVAL_EPISODES,
                        weights=current_weights,
                    )
                    print(f'    평가: success={eval_result["success_rate"]:.3f}, collision={eval_result["collision_rate"]:.3f}')
                    metrics_history.append(eval_result)

                    # 수렴 확인
                    if eval_result["success_rate"] >= TARGET_SUCCESS and eval_result["collision_rate"] <= TARGET_COLLISION:
                        converged = True
                        print(f'    수렴 달성!')
                        break

                    # Plateau 감지 → 가중치 리셋
                    col_plateau = check_plateau(metrics_history, "collision_rate")
                    suc_plateau = check_plateau(metrics_history, "success_rate")
                    if col_plateau and suc_plateau:
                        print(f'    Plateau → 가중치 리셋')
                        current_weights = RewardWeights(
                            collision=-500.0, proximity=-10.0, danger_zone=-20.0,
                            progress=3.0, episode_success=500.0,
                            route_deviation=-0.2, smoothness=-0.05, fuel=-0.02, ice_concentration=-0.3,
                        )
                    else:
                        current_weights = adjust_weights(current_weights, eval_result)

                    # Fine-tuning
                    ft_save = f'{MODELS_DIR}/{key}_realice_iter{iteration}.zip'
                    finetune_on_real_ice(
                        model_path=ft_model_path,
                        route=route, ice_class=ice_class, ship_type=ship_type,
                        real_icebergs=all_real_icebergs,
                        ice_concentration=ice_conc,
                        save_path=ft_save,
                        timesteps=FINETUNE_STEPS,
                        weights=current_weights,
                    )
                    ft_model_path = ft_save

                # 최종 평가
                final_eval = evaluate_on_real_ice(
                    model_path=ft_model_path,
                    route=route, ice_class=ice_class, ship_type=ship_type,
                    real_icebergs=all_real_icebergs,
                    ice_concentration=ice_conc,
                    n_episodes=N_EVAL_EPISODES,
                )
                print(f'  [{key}] 최종: success={final_eval["success_rate"]:.3f}, collision={final_eval["collision_rate"]:.3f}, 수렴={converged}')

                realice_results[key] = {
                    "route": route, "ice_class": ice_class, "ship_type": ship_type,
                    "ice_concentration": ice_conc,
                    "n_real_icebergs": len(all_real_icebergs),
                    "iterations": len(metrics_history),
                    "converged": converged,
                    "metrics_history": metrics_history,
                    "final_eval": final_eval,
                    "ft_model": ft_model_path,
                    "done": True,
                }
                with open(REALICE_FILE, 'w') as f:
                    json.dump(realice_results, f, indent=2)
                done_count += 1

            except Exception as e:
                print(f'  [{key}] 오류: {e}')
                fail_count += 1

            now = time.time()
            if now - last_log_time >= LOG_INTERVAL:
                total_done = done_count + skip_count
                conv = sum(1 for v in realice_results.values() if v.get('converged'))
                print(f'[{time.strftime("%H:%M")}] 진행={total_done}/{total_combinations} 완료={done_count} 수렴={conv} 스킵={skip_count} 실패={fail_count}')
                last_log_time = now

conv_total = sum(1 for v in realice_results.values() if v.get('converged'))
print(f'\n[{time.strftime("%H:%M")}] 완료 | 처리={done_count} 수렴={conv_total}/{total_combinations} 스킵={skip_count} 실패={fail_count}')


In [ ]:
# ── CELL 15: 최종 결과 요약 및 시각화 ───────────────────────
import json, os
import matplotlib.pyplot as plt
import numpy as np

DRIVE_BASE   = '/content/drive/MyDrive/arctic_rl'
RESULTS_DIR  = f'{DRIVE_BASE}/results'
REALICE_FILE = f'{RESULTS_DIR}/realice_results.json'

with open(REALICE_FILE) as f:
    results = json.load(f)

done_results = {k: v for k, v in results.items() if v.get('done')}
print(f'완료된 조합: {len(done_results)}개')

# 항로별 평균 성능
print('\n[항로별 평균 성능 - 실데이터 환경]')
print(f'{"항로":8} {"평가전 성공율":>12} {"평가후 성공율":>12} {"개선":>8}')
print('-' * 45)

for route in ['NSR', 'NWP', 'TSR']:
    route_res = [v for v in done_results.values() if v['route'] == route]
    if not route_res:
        continue
    before = np.mean([v['eval_before']['success_rate'] for v in route_res])
    after_vals = [v['eval_after']['success_rate'] for v in route_res if v.get('eval_after')]
    after = np.mean(after_vals) if after_vals else before
    delta = after - before
    print(f'{route:8} {before:12.3f} {after:12.3f} {delta:+8.3f}')

# 빙급별 평균 성능
print('\n[빙급별 평균 충돌율 - 실데이터 환경]')
print(f'{"빙급":10} {"평가전":>10} {"평가후":>10}')
print('-' * 33)
for ice_class in ['PC7','PC6','PC5','PC4','PC3','IA Super','IA']:
    ic_res = [v for v in done_results.values() if v['ice_class'] == ice_class]
    if not ic_res:
        continue
    before = np.mean([v['eval_before']['collision_rate'] for v in ic_res])
    after_vals = [v['eval_after']['collision_rate'] for v in ic_res if v.get('eval_after')]
    after = np.mean(after_vals) if after_vals else before
    print(f'{ice_class:10} {before:10.3f} {after:10.3f}')

# 시각화
if done_results:
    routes_list = sorted(set(v['route'] for v in done_results.values()))
    before_rates = []
    after_rates  = []
    labels = []
    for route in routes_list:
        rv = [v for v in done_results.values() if v['route'] == route]
        b = np.mean([v['eval_before']['success_rate'] for v in rv])
        a_vals = [v['eval_after']['success_rate'] for v in rv if v.get('eval_after')]
        a = np.mean(a_vals) if a_vals else b
        before_rates.append(b)
        after_rates.append(a)
        labels.append(route)

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - 0.2, before_rates, 0.35, label='Fine-tune 전', color='steelblue')
    bars2 = ax.bar(x + 0.2, after_rates,  0.35, label='Fine-tune 후', color='darkorange')
    ax.set_xlabel('항로')
    ax.set_ylabel('평균 성공율')
    ax.set_title('실제 빙하 데이터 환경 - 항로별 성공율 (Fine-tune 전후)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plot_path = f'{RESULTS_DIR}/realice_performance.png'
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'차트 저장: {plot_path}')

print('\n[완료] 모든 결과가 Drive에 저장되었습니다.')
print(f'  결과 JSON: {REALICE_FILE}')